#**Optimización de la asignación de tickets mediante Constraint Optimization Programming (COP)**


Este notebook implementa el modelo de optimización propuesto en el Trabajo Fin de Máster para la asignación automática de tickets a empleados.

El objetivo consiste en minimizar simultáneamente el tiempo esperado de resolución y favorecer un reparto equilibrado de la carga de trabajo, respetando las restricciones operativas de la empresa.

El modelo se implementa mediante Constraint Optimization Programming utilizando el solver CP-SAT de Google OR-Tools.

## 1. Instalación de dependencias

En esta sección se instalan todas las librerías necesarias para ejecutar el modelo de optimización.

In [1]:
!pip install -q ortools pandas numpy scikit-learn openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 13.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.


## 2. Conexión con Google Drive

Los datos empleados en este estudio se encuentran almacenados en Google Drive.

Por tanto, el primer paso consiste en montar la unidad para poder acceder a los archivos de entrada y guardar posteriormente los resultados obtenidos.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 3. Configuración general

En este apartado se definen las rutas de los archivos y los parámetros generales del problema.

Modificar únicamente esta sección permite reutilizar el algoritmo para otros conjuntos de datos sin necesidad de cambiar el resto del código.

In [3]:
from pathlib import Path

# ============================================================
# RUTAS DE LOS ARCHIVOS EN GOOGLE DRIVE
# ============================================================

# Carpeta principal del proyecto dentro de Google Drive.
# Modifica esta ruta si tu carpeta tiene otro nombre o ubicación.
BASE_DIR = Path("/content/drive/MyDrive/TFM")

# Archivo generado previamente desde R.
INPUT_CSV = Path("/content/drive/MyDrive/tickets_para_optimizacion.csv")

# Carpeta donde se guardarán los resultados del COP.
OUTPUT_DIR = BASE_DIR / "resultados_cop"

# Crear automáticamente la carpeta de resultados si no existe.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# TAMAÑO DE LA INSTANCIA
# ============================================================

# Número de tickets que se desea incluir en la instancia.
# Para el caso de estudio principal se utilizarán 200 tickets.
#
# Ejemplos para el análisis de escalabilidad:
# N_TICKETS = 25
# N_TICKETS = 50
# N_TICKETS = 100
# N_TICKETS = 150
# N_TICKETS = 200
#
# Usa None para incluir todos los tickets disponibles.
N_TICKETS = 200


# ============================================================
# CAPACIDAD DE LOS EMPLEADOS
# ============================================================

# Capacidad semanal por defecto de cada empleado, expresada en horas.
DEFAULT_CAPACITY_HOURS = 40.0

# En caso de que algunos empleados tengan capacidades diferentes,
# podrán especificarse posteriormente mediante un diccionario.
CAPACITY_BY_EMPLOYEE = {}


# ============================================================
# FUNCIÓN OBJETIVO
# ============================================================

# Peso del término asociado al equilibrio de la carga de trabajo.
#
# Un valor pequeño concede mayor importancia al tiempo total.
# Un valor alto concede mayor importancia al reparto equilibrado.
BETA_BALANCE = 1.0


# ============================================================
# RESTRICCIONES ACTIVAS
# ============================================================

# Exigir que el empleado tenga experiencia histórica resolviendo
# tickets con la prioridad correspondiente.
ENFORCE_PRIORITY_HISTORY = True

# Exigir que el ticket y el empleado pertenezcan al mismo equipo.
ENFORCE_TEAM = True

# Exigir que el tiempo esperado de resolución no supere el SLA.
ENFORCE_SLA = True


# ============================================================
# EXPERIENCIA HISTÓRICA POR PRIORIDAD
# ============================================================

# Número mínimo de tickets históricos de una prioridad que debe
# haber resuelto un empleado para considerarlo elegible.
#
# Con valor 1, basta con que haya resuelto al menos un ticket
# de dicha prioridad.
MIN_PRIORITY_HISTORY = 1


# ============================================================
# ACUERDOS DE NIVEL DE SERVICIO
# ============================================================

# Tiempo máximo permitido para resolver un ticket según su prioridad.
# Los valores deben expresarse en horas.
#
# Estos valores son provisionales y deben sustituirse por los SLA
# definitivos definidos para el caso de estudio.
SLA_BY_PRIORITY = {
    "P1": 8.0,
    "P2": 16.0,
    "P3": 24.0,
    "P4": 40.0,
}


# ============================================================
# CONFIGURACIÓN DEL SOLVER
# ============================================================

# Tiempo máximo de ejecución del solver en segundos.
MAX_SOLVER_TIME_SECONDS = 300

# Número de procesadores que podrá utilizar OR-Tools.
NUM_SEARCH_WORKERS = 8

# Escala utilizada para convertir las horas decimales en enteros.
# CP-SAT trabaja con coeficientes enteros.
#
# TIME_SCALE = 100 conserva dos decimales.
TIME_SCALE = 100


# ============================================================
# REPRODUCIBILIDAD
# ============================================================

# Semilla utilizada para seleccionar los tickets y entrenar
# posteriormente el modelo predictivo.
RANDOM_SEED = 42


# ============================================================
# COMPROBACIÓN DE LA CONFIGURACIÓN
# ============================================================

print("Configuración cargada correctamente.")
print("Archivo de entrada:", INPUT_CSV)
print("Carpeta de resultados:", OUTPUT_DIR)
print("Número de tickets:", N_TICKETS)
print("Capacidad semanal por empleado:", DEFAULT_CAPACITY_HOURS, "horas")
print("Peso del equilibrio de carga (beta):", BETA_BALANCE)
print("Restricción por prioridad:", ENFORCE_PRIORITY_HISTORY)
print("Restricción por equipo:", ENFORCE_TEAM)
print("Restricción de SLA:", ENFORCE_SLA)

Configuración cargada correctamente.
Archivo de entrada: /content/drive/MyDrive/tickets_para_optimizacion.csv
Carpeta de resultados: /content/drive/MyDrive/TFM/resultados_cop
Número de tickets: 200
Capacidad semanal por empleado: 40.0 horas
Peso del equilibrio de carga (beta): 1.0
Restricción por prioridad: True
Restricción por equipo: True
Restricción de SLA: True


## 4. Importación de librerías

En esta sección se importan las librerías necesarias para la lectura y manipulación de los datos, la construcción de la matriz de tiempos esperados y la formulación del problema de optimización.

Pandas y NumPy se utilizan para gestionar las estructuras de datos y realizar operaciones matriciales. Scikit-Learn se emplea para construir el modelo predictivo encargado de estimar los tiempos esperados de resolución para cada combinación ticket-empleado. Finalmente, Google OR-Tools proporciona el solver CP-SAT utilizado para formular y resolver el problema de optimización con restricciones.

In [4]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

# Gestión de rutas
from pathlib import Path

# Medición del tiempo de ejecución
from time import perf_counter

# Tipos auxiliares
from typing import Optional

# Operaciones matemáticas
import math

# Manipulación de datos
import numpy as np
import pandas as pd

# Visualización de tablas en Google Colab
from IPython.display import display

# Herramientas de aprendizaje automático
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Solver de optimización
from ortools.sat.python import cp_model

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 5. Definición de funciones auxiliares

Con el objetivo de mejorar la organización del código y facilitar su reutilización, se definen una serie de funciones auxiliares que serán empleadas durante las distintas etapas del proceso de optimización.

Estas funciones realizan tareas de apoyo como la detección automática de las variables necesarias dentro del conjunto de datos, la normalización de determinados valores y la validación de la información antes de construir el modelo matemático. De esta forma, el algoritmo puede adaptarse a conjuntos de datos de estructura similar sin necesidad de modificar el resto de la implementación.

La utilización de funciones independientes contribuye además a mejorar la legibilidad del código, simplificar su mantenimiento y favorecer la reproducibilidad de los experimentos realizados.

In [5]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def normalizar_booleano(serie):
    """
    Convierte diferentes representaciones de valores lógicos
    (True, False, Yes, No, 1, 0, etc.) en valores booleanos.
    """

    if serie.dtype == bool:
        return serie

    return (
        serie.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
            "si": True,
            "sí": True
        })
        .fillna(False)
    )


def primera_columna_existente(df, candidatos):
    """
    Busca la primera columna existente dentro de una lista
    de posibles nombres.

    Esto permite utilizar conjuntos de datos cuya estructura
    sea ligeramente distinta.
    """

    columnas = {
        columna.lower().strip(): columna
        for columna in df.columns
    }

    for candidato in candidatos:
        candidato = candidato.lower().strip()

        if candidato in columnas:
            return columnas[candidato]

    return None


def detectar_columnas(df):
    """
    Detecta automáticamente las columnas necesarias para
    construir el problema de optimización.
    """

    columnas = {

        "ticket": primera_columna_existente(df, [
            "Issue key",
            "Issue Key",
            "Ticket",
            "ticket_id"
        ]),

        "employee": primera_columna_existente(df, [
            "Assignee",
            "Empleado",
            "Employee"
        ]),

        "team": primera_columna_existente(df, [
            "Team",
            "Equipo",
            "Department"
        ]),

        "priority": primera_columna_existente(df, [
            "Priority",
            "Priority level"
        ]),

        "worked_hours": primera_columna_existente(df, [
            "Worked hours",
            "Time to resolution en Horas"
        ]),

        "issue_type": primera_columna_existente(df, [
            "Issue Type"
        ]),

        "request_type": primera_columna_existente(df, [
            "Request Type"
        ])
    }

    # Variables obligatorias para continuar
    obligatorias = [
        "ticket",
        "employee",
        "priority",
        "worked_hours"
    ]

    faltantes = [
        variable
        for variable in obligatorias
        if columnas[variable] is None
    ]

    if len(faltantes) > 0:

        raise ValueError(
            "No se encontraron las siguientes columnas obligatorias:\n\n"
            + "\n".join(faltantes)
        )

    return columnas

## 6. Lectura y preparación del conjunto de datos

Una vez definidas las funciones auxiliares, se procede a cargar el conjunto de datos obtenido tras el proceso de preprocesado realizado en R. Este conjunto constituye la base histórica utilizada para construir la instancia del problema de optimización y contiene exclusivamente observaciones válidas, tras la eliminación de registros incompletos y valores atípicos realizada durante la fase de análisis exploratorio.

Posteriormente, se lleva a cabo una validación inicial de la información. En esta etapa se detectan automáticamente las columnas necesarias para el modelo, se eliminan registros con información incompleta y se normaliza el formato de determinadas variables con el objetivo de garantizar la consistencia de los datos antes de comenzar el proceso de optimización.

Como resultado de esta fase se obtiene un conjunto de datos limpio y estructurado que servirá como entrada para la construcción de la instancia del problema y la estimación de los parámetros del modelo matemático.

In [6]:
# ============================================================
# LECTURA Y PREPARACIÓN DE LOS DATOS
# ============================================================

# Leer el conjunto de datos generado durante el preprocesado en R
df_raw = pd.read_csv(INPUT_CSV)

print("Dimensiones originales del conjunto de datos:")
print(df_raw.shape)

print("\nPrimeras observaciones:")
display(df_raw.head())


# ============================================================
# DETECCIÓN AUTOMÁTICA DE VARIABLES
# ============================================================

columnas = detectar_columnas(df_raw)

print("\nColumnas detectadas automáticamente:")

for nombre, columna in columnas.items():
    print(f"{nombre:15}: {columna}")


# ============================================================
# COPIA DE TRABAJO
# ============================================================

df = df_raw.copy()


# ============================================================
# ELIMINACIÓN DE REGISTROS INCOMPLETOS
# ============================================================

variables_obligatorias = [
    columnas["employee"],
    columnas["priority"],
    columnas["worked_hours"]
]

df = df.dropna(subset=variables_obligatorias)


# ============================================================
# NORMALIZACIÓN DE VARIABLES
# ============================================================

# Empleado
df[columnas["employee"]] = (
    df[columnas["employee"]]
    .astype(str)
    .str.strip()
)

# Prioridad
df[columnas["priority"]] = (
    df[columnas["priority"]]
    .astype(str)
    .str.strip()
)

# Tiempo de resolución
df[columnas["worked_hours"]] = pd.to_numeric(
    df[columnas["worked_hours"]],
    errors="coerce"
)


# ============================================================
# ELIMINACIÓN DE OBSERVACIONES INVÁLIDAS
# ============================================================

df = df[
    (df[columnas["employee"]] != "")
    &
    (df[columnas["worked_hours"]].notna())
    &
    (df[columnas["worked_hours"]] > 0)
].copy()


# ============================================================
# CREACIÓN DE VARIABLES AUXILIARES
# ============================================================

# Si no existe identificador del ticket,
# se genera automáticamente.

if columnas["ticket"] is None:

    df["ticket_id_generado"] = [
        f"T{i+1:06d}"
        for i in range(len(df))
    ]

    columnas["ticket"] = "ticket_id_generado"


# Si no existe equipo,
# todos los registros pertenecen al mismo.

if columnas["team"] is None:

    df["Team_generado"] = "GENERAL"

    columnas["team"] = "Team_generado"


# Normalizar nombre del equipo

df[columnas["team"]] = (
    df[columnas["team"]]
    .fillna("SIN_EQUIPO")
    .astype(str)
    .str.strip()
)


# ============================================================
# INFORMACIÓN FINAL
# ============================================================

print("\n------------------------------------------")
print("Datos preparados correctamente")
print("------------------------------------------")

print("Número de registros:", len(df))

print("Número de tickets:",
      df[columnas["ticket"]].nunique())

print("Número de empleados:",
      df[columnas["employee"]].nunique())

print("Número de equipos:",
      df[columnas["team"]].nunique())

print("Prioridades detectadas:",
      sorted(df[columnas["priority"]].unique()))

print("\nVista previa del conjunto de datos limpio:")

display(df.head())

Dimensiones originales del conjunto de datos:
(9328, 19)

Primeras observaciones:


,id_fila_original,Issue key,Assignee,Team,Issue Type,Request Type,Priority,Priority level,Worked hours,Created-Update en Horas,Inward Tickets,Outward Tickets,Inward Project,Outward Project,Inward,Outward,Total Link,Status,Resolution
0,1,DIG-72,Jonas De Jonge,SEC,[System] Service request,Get something new from IT,P3,2,30.984722,5856,0,0,0,0,0,0,0,Canceled,Done
1,2,DIG-218,Louis Pintjens,CS,[System] Incident,Get help from the Customer Success Team (Incid...,P4,1,221.166111,7584,0,1,0,0,0,1,1,Close without notification,Done
2,3,DIG-228,Louis Pintjens,CS,[System] Incident,Get help from the Customer Success Team (Incid...,P4,1,221.132778,1248,0,0,0,0,0,0,0,Close without notification,Done
3,4,DIG-236,Louis Pintjens,CS,[System] Incident,Get help from the Customer Success Team (Incid...,P3,2,221.288889,7584,0,1,0,0,0,1,1,Close without notification,Done
4,5,DIG-454,Gert Ivens,IT,[System] Service request,Onboard New MPL Employees,P4,1,47.444722,1944,0,0,0,0,0,0,0,Close without notification,Done



Columnas detectadas automáticamente:
ticket         : Issue key
employee       : Assignee
team           : Team
priority       : Priority
worked_hours   : Worked hours
issue_type     : Issue Type
request_type   : Request Type

------------------------------------------
Datos preparados correctamente
------------------------------------------
Número de registros: 9326
Número de tickets: 9326
Número de empleados: 40
Número de equipos: 7
Prioridades detectadas: ['P1', 'P2', 'P3', 'P4']

Vista previa del conjunto de datos limpio:


,id_fila_original,Issue key,Assignee,Team,Issue Type,Request Type,Priority,Priority level,Worked hours,Created-Update en Horas,Inward Tickets,Outward Tickets,Inward Project,Outward Project,Inward,Outward,Total Link,Status,Resolution
0,1,DIG-72,Jonas De Jonge,SEC,[System] Service request,Get something new from IT,P3,2,30.984722,5856,0,0,0,0,0,0,0,Canceled,Done
1,2,DIG-218,Louis Pintjens,CS,[System] Incident,Get help from the Customer Success Team (Incid...,P4,1,221.166111,7584,0,1,0,0,0,1,1,Close without notification,Done
2,3,DIG-228,Louis Pintjens,CS,[System] Incident,Get help from the Customer Success Team (Incid...,P4,1,221.132778,1248,0,0,0,0,0,0,0,Close without notification,Done
3,4,DIG-236,Louis Pintjens,CS,[System] Incident,Get help from the Customer Success Team (Incid...,P3,2,221.288889,7584,0,1,0,0,0,1,1,Close without notification,Done
4,5,DIG-454,Gert Ivens,IT,[System] Service request,Onboard New MPL Employees,P4,1,47.444722,1944,0,0,0,0,0,0,0,Close without notification,Done


## 7. Construcción de la instancia del problema

Una vez preparado el conjunto de datos, se construye la instancia del problema de optimización. Para ello se seleccionan los tickets que formarán parte del experimento, se identifican los empleados disponibles y se generan los conjuntos necesarios para la formulación matemática.

Con el fin de facilitar el análisis de escalabilidad, el número de tickets considerados puede modificarse mediante el parámetro `N_TICKETS`. Esta característica permite resolver instancias de diferentes tamaños sin necesidad de alterar el resto del algoritmo.

Durante esta etapa también se generan las estructuras de datos que servirán de base para la formulación del modelo matemático, incluyendo los conjuntos de tickets y empleados, así como la capacidad disponible de cada trabajador.

In [7]:
# ============================================================
# 7. CONSTRUCCIÓN DE LA INSTANCIA DEL PROBLEMA
# ============================================================

print("Construyendo la instancia de optimización...")


# ------------------------------------------------------------
# 7.1. Selección de los tickets de la instancia
# ------------------------------------------------------------

# Se parte de toda la base histórica limpia.
tickets_df = df.copy()

# Si N_TICKETS contiene un número, se selecciona una muestra
# aleatoria reproducible de ese tamaño.
#
# Si N_TICKETS es None, se utilizan todos los tickets.

if N_TICKETS is not None:

    number_of_selected_tickets = min(
        int(N_TICKETS),
        len(tickets_df)
    )

    tickets_df = (
        tickets_df
        .sample(
            n=number_of_selected_tickets,
            random_state=RANDOM_SEED
        )
        .reset_index(drop=True)
    )

else:

    tickets_df = (
        tickets_df
        .reset_index(drop=True)
    )


print(
    "Tickets seleccionados:",
    len(tickets_df)
)


# ------------------------------------------------------------
# 7.2. Comprobación de identificadores duplicados
# ------------------------------------------------------------

# La formulación requiere que cada ticket tenga un identificador
# único. Si existen duplicados, se genera un identificador interno
# que conserva el identificador original y añade el número de fila.

ticket_column_original = columnas["ticket"]

tickets_df[
    "optimization_ticket_id"
] = (
    tickets_df[ticket_column_original]
    .astype(str)
    .str.strip()
)


if tickets_df[
    "optimization_ticket_id"
].duplicated().any():

    print(
        "Advertencia: existen identificadores de ticket "
        "duplicados en la instancia."
    )

    tickets_df[
        "optimization_ticket_id"
    ] = [
        f"{ticket_id}__{row_number}"
        for row_number, ticket_id
        in enumerate(
            tickets_df[
                "optimization_ticket_id"
            ],
            start=1
        )
    ]


# Esta será la columna utilizada internamente por el COP.
optimization_ticket_column = (
    "optimization_ticket_id"
)


# ------------------------------------------------------------
# 7.3. Obtención de todos los empleados de la base histórica
# ------------------------------------------------------------

# Los empleados no se extraen únicamente de tickets_df.
# Se obtienen de toda la base histórica limpia, de forma que
# todos los recursos disponibles puedan ser considerados.

employees = sorted(
    df[columnas["employee"]]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[
        lambda employee_names:
        employee_names != ""
    ]
    .unique()
    .tolist()
)


print(
    "Empleados detectados en toda la base:",
    len(employees)
)


# ------------------------------------------------------------
# 7.4. Conjuntos principales T y E
# ------------------------------------------------------------

# T: conjunto de tickets de la instancia.
ticket_ids = (
    tickets_df[
        optimization_ticket_column
    ]
    .astype(str)
    .tolist()
)

# E: conjunto de empleados disponibles.
employee_ids = employees.copy()


# ------------------------------------------------------------
# 7.5. Índices numéricos auxiliares
# ------------------------------------------------------------

ticket_to_index = {
    ticket: i
    for i, ticket in enumerate(ticket_ids)
}

employee_to_index = {
    employee: j
    for j, employee in enumerate(employees)
}


# ------------------------------------------------------------
# 7.6. Capacidad disponible de cada empleado
# ------------------------------------------------------------

# Cada empleado recibe la capacidad general, salvo que se haya
# definido un valor específico en CAPACITY_BY_EMPLOYEE.

employee_capacity = {
    employee: float(
        CAPACITY_BY_EMPLOYEE.get(
            employee,
            DEFAULT_CAPACITY_HOURS
        )
    )
    for employee in employees
}


# Validación básica de capacidades.

invalid_employee_capacities = {
    employee: capacity
    for employee, capacity
    in employee_capacity.items()
    if (
        capacity is None
        or not np.isfinite(capacity)
        or capacity <= 0
    )
}

if invalid_employee_capacities:

    raise ValueError(
        "Existen empleados con capacidades inválidas:\n"
        + "\n".join(
            f"{employee}: {capacity}"
            for employee, capacity
            in invalid_employee_capacities.items()
        )
    )


# ------------------------------------------------------------
# 7.7. Prioridad de cada ticket
# ------------------------------------------------------------

ticket_priority = dict(
    zip(
        tickets_df[
            optimization_ticket_column
        ],
        tickets_df[
            columnas["priority"]
        ]
        .astype(str)
        .str.strip()
        .str.upper()
    )
)


# ------------------------------------------------------------
# 7.8. Equipo responsable de cada ticket
# ------------------------------------------------------------

ticket_team = dict(
    zip(
        tickets_df[
            optimization_ticket_column
        ],
        tickets_df[
            columnas["team"]
        ]
        .fillna("SIN_EQUIPO")
        .astype(str)
        .str.strip()
        .str.upper()
    )
)


# ------------------------------------------------------------
# 7.9. Equipo principal de cada empleado
# ------------------------------------------------------------

# El equipo de cada empleado se obtiene utilizando toda la base
# histórica y tomando el valor más frecuente.

employee_team = (
    df
    .assign(
        __employee_clean=(
            df[columnas["employee"]]
            .astype(str)
            .str.strip()
        ),
        __team_clean=(
            df[columnas["team"]]
            .fillna("SIN_EQUIPO")
            .astype(str)
            .str.strip()
            .str.upper()
        )
    )
    .loc[
        lambda data:
        data["__employee_clean"] != ""
    ]
    .groupby(
        "__employee_clean"
    )["__team_clean"]
    .agg(
        lambda values: (
            values.mode().iloc[0]
            if not values.mode().empty
            else "SIN_EQUIPO"
        )
    )
    .to_dict()
)


# Garantizar que todos los empleados tengan un equipo asignado.

for employee in employees:

    if employee not in employee_team:

        employee_team[employee] = (
            "SIN_EQUIPO"
        )


# ------------------------------------------------------------
# 7.10. Correspondencia con el identificador original
# ------------------------------------------------------------

# Esta tabla permitirá recuperar posteriormente el identificador
# original del ticket en los resultados.

ticket_reference_df = tickets_df[[
    optimization_ticket_column,
    ticket_column_original
]].copy()

ticket_reference_df.columns = [
    "optimization_ticket_id",
    "original_ticket_id"
]


# ------------------------------------------------------------
# 7.11. Resumen de la instancia
# ------------------------------------------------------------

instance_summary = pd.DataFrame({
    "Indicador": [
        "Tickets disponibles en la base limpia",
        "Tickets seleccionados para la instancia",
        "Empleados presentes en toda la base",
        "Equipos presentes en la instancia",
        "Equipos presentes en toda la base",
        "Prioridades presentes en la instancia",
        "Capacidad semanal por defecto",
        "Capacidad total disponible"
    ],
    "Valor": [
        len(df),
        len(ticket_ids),
        len(employees),
        tickets_df[
            columnas["team"]
        ].nunique(),
        df[
            columnas["team"]
        ].nunique(),
        tickets_df[
            columnas["priority"]
        ].nunique(),
        DEFAULT_CAPACITY_HOURS,
        sum(
            employee_capacity.values()
        )
    ]
})


print("\n------------------------------------------")
print("RESUMEN DE LA INSTANCIA")
print("------------------------------------------")

display(instance_summary)


# ------------------------------------------------------------
# 7.12. Vista previa de tickets seleccionados
# ------------------------------------------------------------

preview_columns = [
    optimization_ticket_column,
    ticket_column_original,
    columnas["employee"],
    columnas["priority"],
    columnas["team"],
    columnas["worked_hours"]
]

# Evitar mostrar dos veces la misma columna si el identificador
# original ya se llama optimization_ticket_id.

preview_columns = list(
    dict.fromkeys(preview_columns)
)


print("\nVista previa de los tickets seleccionados:")

display(
    tickets_df[
        preview_columns
    ].head(10)
)


# ------------------------------------------------------------
# 7.13. Vista previa de empleados
# ------------------------------------------------------------

employees_df = pd.DataFrame({
    "employee": employees,
    "team": [
        employee_team.get(
            employee,
            "SIN_EQUIPO"
        )
        for employee in employees
    ],
    "capacity_hours": [
        employee_capacity[employee]
        for employee in employees
    ]
})


print("\nVista previa de empleados disponibles:")

display(
    employees_df.head(20)
)


# ------------------------------------------------------------
# 7.14. Comprobación final
# ------------------------------------------------------------

historical_employee_count = (
    df[columnas["employee"]]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[
        lambda employee_names:
        employee_names != ""
    ]
    .nunique()
)


print(
    "\nEmpleados únicos en toda la base:",
    historical_employee_count
)

print(
    "Empleados incluidos en el COP:",
    len(employees)
)


if historical_employee_count != len(employees):

    print(
        "Advertencia: el número no coincide. "
        "Revise posibles diferencias de escritura, "
        "espacios o valores nulos."
    )

else:

    print(
        "✓ Todos los empleados válidos de la base "
        "han sido incluidos en la instancia."
    )

Construyendo la instancia de optimización...
Tickets seleccionados: 200
Empleados detectados en toda la base: 40

------------------------------------------
RESUMEN DE LA INSTANCIA
------------------------------------------


,Indicador,Valor
0,Tickets disponibles en la base limpia,9326.0
1,Tickets seleccionados para la instancia,200.0
2,Empleados presentes en toda la base,40.0
3,Equipos presentes en la instancia,5.0
4,Equipos presentes en toda la base,7.0
5,Prioridades presentes en la instancia,4.0
6,Capacidad semanal por defecto,40.0
7,Capacidad total disponible,1600.0



Vista previa de los tickets seleccionados:


,optimization_ticket_id,Issue key,Assignee,Priority,Team,Worked hours
0,DIG-23000,DIG-23000,Gert Ivens,P3,IT,1.972222
1,DIG-11893,DIG-11893,Andy Van Puyvelde,P3,BCFIN,28.742222
2,DIG-20407,DIG-20407,Papitchaya Paonta,P3,CS,0.771389
3,DIG-16326,DIG-16326,Jef Bogaerts,P3,IT,0.798889
4,DIG-21635,DIG-21635,Louis Pintjens,P3,CS,0.771667
5,DIG-4349,DIG-4349,Christophe Kokken,P3,CS,43.095278
6,DIG-18650,DIG-18650,Christophe Kokken,P3,CS,0.699444
7,DIG-18391,DIG-18391,Louis Pintjens,P3,CS,0.791389
8,DIG-14124,DIG-14124,Jennie Gouveia,P3,CS,23.855833
9,DIG-16006,DIG-16006,Jennie Gouveia,P3,CS,0.445000



Vista previa de empleados disponibles:


,employee,team,capacity_hours
0,Andre Velicev,CS,40.0
1,Andy Van Puyvelde,BCFIN,40.0
2,Anthony Restiau,CS,40.0
3,Anthony Roels,CS,40.0
4,Ayoub Azzouz,CRM,40.0
5,Bart Jansen,BCFIN,40.0
6,Bjorn Van Cotthem,DW,40.0
7,Camila Reis,CS,40.0
8,Carolina Vaeza,BCFIN,40.0
9,Christophe Kokken,CS,40.0



Empleados únicos en toda la base: 40
Empleados incluidos en el COP: 40
✓ Todos los empleados válidos de la base han sido incluidos en la instancia.


## 8. Estimación de la matriz de tiempos esperados

El modelo de optimización requiere conocer el tiempo esperado que emplearía cada trabajador en resolver cada uno de los tickets incluidos en la instancia. Sin embargo, los datos históricos únicamente contienen el tiempo correspondiente al empleado que resolvió realmente cada incidencia, por lo que no se dispone directamente del tiempo que habrían empleado los demás trabajadores.

Para estimar estos valores se entrena un modelo de Random Forest utilizando los tickets históricos. La variable objetivo corresponde a las horas trabajadas, mientras que las variables explicativas incluyen las características del ticket y la identidad del empleado responsable de su resolución.

Una vez entrenado el modelo, se generan todas las combinaciones posibles entre los tickets de la instancia y los empleados disponibles. Para cada combinación ticket-empleado, el modelo predice el tiempo esperado de resolución. Estas estimaciones constituyen la matriz \(t_{ij}\), donde cada elemento representa el tiempo esperado que necesitaría el empleado \(j\) para resolver el ticket \(i\).

Con el objetivo de evitar una posible filtración de información, los tickets seleccionados para la instancia de optimización se excluyen del conjunto utilizado para entrenar el modelo predictivo. De esta forma, sus tiempos estimados se obtienen a partir del comportamiento observado en otros tickets históricos.

In [8]:
# ============================================================
# ESTIMACIÓN DE LA MATRIZ DE TIEMPOS t_ij
# ============================================================

print("Preparando el modelo predictivo de tiempos...")


# ------------------------------------------------------------
# Definición de la variable objetivo
# ------------------------------------------------------------

target_column = columnas["worked_hours"]


# ------------------------------------------------------------
# Selección de variables explicativas
# ------------------------------------------------------------

categorical_features = [
    columnas["employee"],
    columnas["priority"],
    columnas["team"]
]

# Añadir variables opcionales cuando estén disponibles
if columnas["issue_type"] is not None:
    categorical_features.append(columnas["issue_type"])

if columnas["request_type"] is not None:
    categorical_features.append(columnas["request_type"])

# Eliminar posibles duplicados
categorical_features = list(dict.fromkeys(categorical_features))


print("\nVariables categóricas utilizadas:")

for feature in categorical_features:
    print("-", feature)


# ------------------------------------------------------------
# Creación del conjunto de entrenamiento
# ------------------------------------------------------------

# Se excluyen del entrenamiento los tickets seleccionados
# para la instancia de optimización.

selected_ticket_ids = set(
    tickets_df[columnas["ticket"]]
)

training_df = df[
    ~df[columnas["ticket"]].isin(selected_ticket_ids)
].copy()


# En caso de que queden muy pocos registros, se utilizará
# el conjunto histórico completo.

if len(training_df) < 100:

    print(
        "\nAdvertencia: quedan pocos registros tras excluir "
        "los tickets de la instancia."
    )

    print("Se utilizará el conjunto histórico completo.")

    training_df = df.copy()


# Mantener únicamente registros completos
training_df = training_df.dropna(
    subset=categorical_features + [target_column]
).copy()


# Garantizar que la variable objetivo sea numérica
training_df[target_column] = pd.to_numeric(
    training_df[target_column],
    errors="coerce"
)

training_df = training_df[
    training_df[target_column].notna()
    &
    (training_df[target_column] > 0)
].copy()


print("\nRegistros utilizados para entrenar:",
      len(training_df))


# ------------------------------------------------------------
# Separación de variables predictoras y objetivo
# ------------------------------------------------------------

X_train = training_df[categorical_features].copy()

y_train = training_df[target_column].copy()


# Convertir variables categóricas a texto
for column in categorical_features:

    X_train[column] = (
        X_train[column]
        .fillna("DESCONOCIDO")
        .astype(str)
        .str.strip()
    )


# ------------------------------------------------------------
# Transformación de variables categóricas
# ------------------------------------------------------------

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ],
    remainder="drop"
)


# ------------------------------------------------------------
# Definición del modelo Random Forest
# ------------------------------------------------------------

random_forest = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=RANDOM_SEED,
    n_jobs=-1
)


# ------------------------------------------------------------
# Creación del pipeline completo
# ------------------------------------------------------------

time_prediction_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", random_forest)
    ]
)


# ------------------------------------------------------------
# Entrenamiento del modelo
# ------------------------------------------------------------

start_training = perf_counter()

time_prediction_model.fit(
    X_train,
    y_train
)

training_time = perf_counter() - start_training


print(
    f"\nModelo entrenado correctamente en "
    f"{training_time:.2f} segundos."
)


# ============================================================
# GENERACIÓN DE COMBINACIONES TICKET-EMPLEADO
# ============================================================

print("\nGenerando combinaciones ticket-empleado...")


prediction_rows = []


for _, ticket_row in tickets_df.iterrows():

    for employee in employees:

        new_row = {}

        # Copiar las características propias del ticket
        for feature in categorical_features:

            if feature == columnas["employee"]:

                # Sustituir el empleado histórico por el empleado
                # cuya duración se desea estimar.
                new_row[feature] = employee

            else:

                value = ticket_row.get(
                    feature,
                    "DESCONOCIDO"
                )

                if pd.isna(value):
                    value = "DESCONOCIDO"

                new_row[feature] = str(value).strip()

        # Conservar el identificador del ticket
        new_row[columnas["ticket"]] = ticket_row[
            columnas["ticket"]
        ]

        prediction_rows.append(new_row)


prediction_df = pd.DataFrame(prediction_rows)


print(
    "Número de combinaciones generadas:",
    len(prediction_df)
)

print(
    "Resultado esperado:",
    len(ticket_ids) * len(employees)
)


# ============================================================
# PREDICCIÓN DE LOS TIEMPOS
# ============================================================

start_prediction = perf_counter()

predicted_hours = time_prediction_model.predict(
    prediction_df[categorical_features]
)

prediction_time = perf_counter() - start_prediction


# Evitar valores negativos o excesivamente próximos a cero
predicted_hours = np.maximum(
    predicted_hours,
    0.01
)


prediction_df["predicted_hours"] = predicted_hours


print(
    f"\nPredicciones realizadas en "
    f"{prediction_time:.2f} segundos."
)


# ============================================================
# CREACIÓN DE LA MATRIZ t_ij
# ============================================================

time_matrix_df = prediction_df.pivot(
    index=columnas["ticket"],
    columns=columnas["employee"],
    values="predicted_hours"
)


# Mantener el orden original de tickets y empleados
time_matrix_df = time_matrix_df.reindex(
    index=ticket_ids,
    columns=employees
)


# Diccionario que utilizará posteriormente el modelo COP
expected_time = {

    (ticket, employee): float(
        time_matrix_df.loc[ticket, employee]
    )

    for ticket in ticket_ids

    for employee in employees
}


# ============================================================
# RESUMEN DE LA MATRIZ
# ============================================================

print("\n------------------------------------------")
print("Matriz de tiempos estimados construida")
print("------------------------------------------")

print("Número de tickets:", len(ticket_ids))

print("Número de empleados:", len(employees))

print("Dimensiones de la matriz:",
      time_matrix_df.shape)

print(
    "Tiempo esperado mínimo:",
    round(time_matrix_df.min().min(), 2),
    "horas"
)

print(
    "Tiempo esperado medio:",
    round(time_matrix_df.stack().mean(), 2),
    "horas"
)

print(
    "Tiempo esperado máximo:",
    round(time_matrix_df.max().max(), 2),
    "horas"
)


print("\nVista previa de la matriz t_ij:")

display(
    time_matrix_df.iloc[:10, :10].round(2)
)

Preparando el modelo predictivo de tiempos...

Variables categóricas utilizadas:
- Assignee
- Priority
- Team
- Issue Type
- Request Type

Registros utilizados para entrenar: 9126

Modelo entrenado correctamente en 13.48 segundos.

Generando combinaciones ticket-empleado...
Número de combinaciones generadas: 8000
Resultado esperado: 8000

Predicciones realizadas en 0.23 segundos.

------------------------------------------
Matriz de tiempos estimados construida
------------------------------------------
Número de tickets: 200
Número de empleados: 40
Dimensiones de la matriz: (200, 40)
Tiempo esperado mínimo: 0.58 horas
Tiempo esperado medio: 25.38 horas
Tiempo esperado máximo: 190.87 horas

Vista previa de la matriz t_ij:


Assignee,Andre Velicev,Andy Van Puyvelde,Anthony Restiau,Anthony Roels,Ayoub Azzouz,Bart Jansen,Bjorn Van Cotthem,Camila Reis,Carolina Vaeza,Christophe Kokken
Issue key,,,,,,,,,,
DIG-23000,5.75,6.71,4.31,17.56,4.31,4.31,4.86,4.31,4.31,21.06
DIG-11893,24.90,20.51,17.13,35.38,17.13,8.33,19.78,17.13,17.13,23.29
DIG-20407,26.89,18.15,12.25,35.38,12.25,9.66,16.50,12.25,12.25,23.38
DIG-16326,14.44,14.73,13.21,22.69,13.21,13.17,13.29,13.21,13.21,33.71
DIG-21635,26.89,18.15,12.25,35.38,12.25,9.66,16.50,12.25,12.25,23.38
DIG-4349,26.89,18.15,12.25,35.38,12.25,9.66,16.50,12.25,12.25,23.38
DIG-18650,26.89,18.15,12.25,35.38,12.25,9.66,16.50,12.25,12.25,23.38
DIG-18391,26.89,18.15,12.25,35.38,12.25,9.66,16.50,12.25,12.25,23.38
DIG-14124,26.89,18.15,12.25,35.38,12.25,9.66,16.50,12.25,12.25,23.38


## 9. Construcción de las restricciones de elegibilidad

Antes de formular el modelo de optimización, se determina qué empleados pueden ser asignados a cada ticket. Esta etapa permite incorporar las reglas operativas del problema mediante la construcción de una matriz de elegibilidad.

La elegibilidad de una combinación ticket-empleado puede depender de tres criterios. En primer lugar, se comprueba si el empleado pertenece al equipo responsable del ticket. En segundo lugar, se verifica que disponga de experiencia histórica resolviendo incidencias de la misma prioridad. Finalmente, se analiza si el tiempo esperado de resolución estimado para esa combinación cumple el acuerdo de nivel de servicio correspondiente.

Una combinación se considera válida únicamente cuando satisface todas las restricciones que se encuentren activadas en la configuración general. Las asignaciones no elegibles quedarán excluidas del modelo matemático y, por tanto, no podrán ser seleccionadas por el solver.

Para evitar que un ticket quede sin candidatos debido a restricciones excesivamente estrictas, se incorpora un mecanismo de respaldo. Cuando ningún empleado satisface simultáneamente todos los criterios, se habilita al trabajador compatible con el equipo que presente el menor tiempo esperado. Si tampoco existen empleados del mismo equipo, se selecciona el empleado con el menor tiempo esperado entre todos los disponibles. Estas situaciones quedan registradas para su posterior revisión.

In [9]:
# ============================================================
# CONSTRUCCIÓN DE LAS RESTRICCIONES DE ELEGIBILIDAD
# ============================================================

print("Construyendo matriz de elegibilidad...")


# ------------------------------------------------------------
# Normalización de prioridades y equipos
# ------------------------------------------------------------

df[columnas["priority"]] = (
    df[columnas["priority"]]
    .astype(str)
    .str.strip()
    .str.upper()
)

tickets_df[columnas["priority"]] = (
    tickets_df[columnas["priority"]]
    .astype(str)
    .str.strip()
    .str.upper()
)

df[columnas["team"]] = (
    df[columnas["team"]]
    .fillna("SIN_EQUIPO")
    .astype(str)
    .str.strip()
    .str.upper()
)

tickets_df[columnas["team"]] = (
    tickets_df[columnas["team"]]
    .fillna("SIN_EQUIPO")
    .astype(str)
    .str.strip()
    .str.upper()
)


# Reconstruir los diccionarios después de normalizar los datos
ticket_priority = dict(
    zip(
        tickets_df[columnas["ticket"]],
        tickets_df[columnas["priority"]]
    )
)

ticket_team = dict(
    zip(
        tickets_df[columnas["ticket"]],
        tickets_df[columnas["team"]]
    )
)


# ------------------------------------------------------------
# Determinación del equipo principal de cada empleado
# ------------------------------------------------------------

employee_team = (
    training_df
    .groupby(columnas["employee"])[columnas["team"]]
    .agg(
        lambda values: (
            values.mode().iloc[0]
            if not values.mode().empty
            else "SIN_EQUIPO"
        )
    )
    .to_dict()
)


# ------------------------------------------------------------
# Experiencia histórica por prioridad
# ------------------------------------------------------------

priority_history = (
    training_df                          # use training_df instead of df
    .groupby([columnas["employee"], columnas["priority"]])
    .size()
    .to_dict()
)


def tiene_experiencia_prioridad(employee, priority):
    """
    Comprueba si el empleado ha resuelto históricamente
    el número mínimo requerido de tickets con una prioridad.
    """

    historical_count = priority_history.get(
        (employee, priority),
        0
    )

    return historical_count >= MIN_PRIORITY_HISTORY


# ------------------------------------------------------------
# Obtención del SLA correspondiente a cada prioridad
# ------------------------------------------------------------

def obtener_sla(priority):
    """
    Devuelve el SLA asociado a una prioridad.

    Cuando la prioridad no aparece en el diccionario configurado,
    se devuelve infinito para no excluir automáticamente
    la combinación.
    """

    priority = str(priority).strip().upper()

    return float(
        SLA_BY_PRIORITY.get(
            priority,
            np.inf
        )
    )


# ------------------------------------------------------------
# Evaluación de cada combinación ticket-empleado
# ------------------------------------------------------------

eligibility = {}

eligibility_details = []

for ticket in ticket_ids:

    priority = ticket_priority[ticket]
    team = ticket_team[ticket]
    sla_hours = obtener_sla(priority)

    for employee in employees:

        predicted_time = expected_time[
            (ticket, employee)
        ]

        employee_assigned_team = employee_team.get(
            employee,
            "SIN_EQUIPO"
        )

        # Restricción de equipo
        team_ok = (
            employee_assigned_team == team
            if ENFORCE_TEAM
            else True
        )

        # Restricción de experiencia por prioridad
        priority_ok = (
            tiene_experiencia_prioridad(
                employee,
                priority
            )
            if ENFORCE_PRIORITY_HISTORY
            else True
        )

        # Restricción de SLA
        sla_ok = (
            predicted_time <= sla_hours
            if ENFORCE_SLA
            else True
        )

        # La combinación es elegible cuando cumple
        # todas las restricciones activas
        is_eligible = (
            team_ok
            and priority_ok
            and sla_ok
        )

        eligibility[(ticket, employee)] = is_eligible

        eligibility_details.append({
            "ticket": ticket,
            "employee": employee,
            "priority": priority,
            "ticket_team": team,
            "employee_team": employee_assigned_team,
            "predicted_hours": predicted_time,
            "sla_hours": sla_hours,
            "team_ok": team_ok,
            "priority_ok": priority_ok,
            "sla_ok": sla_ok,
            "eligible": is_eligible,
            "fallback_applied": False
        })


eligibility_df = pd.DataFrame(
    eligibility_details
)


# ============================================================
# MECANISMO DE RESPALDO PARA TICKETS SIN EMPLEADOS ELEGIBLES
# ============================================================

fallback_records = []

for ticket in ticket_ids:

    eligible_employees = [
        employee
        for employee in employees
        if eligibility[(ticket, employee)]
    ]

    # Si el ticket ya tiene al menos un candidato,
    # no es necesario aplicar el mecanismo de respaldo
    if eligible_employees:
        continue

    ticket_required_team = ticket_team[ticket]

    # Primer nivel de respaldo:
    # empleados pertenecientes al mismo equipo
    same_team_employees = [
        employee
        for employee in employees
        if employee_team.get(
            employee,
            "SIN_EQUIPO"
        ) == ticket_required_team
    ]

    if same_team_employees:

        fallback_employee = min(
            same_team_employees,
            key=lambda employee: expected_time[
                (ticket, employee)
            ]
        )

        fallback_reason = (
            "Empleado más rápido del mismo equipo"
        )

    else:

        # Segundo nivel de respaldo:
        # empleado más rápido entre todos los disponibles
        fallback_employee = min(
            employees,
            key=lambda employee: expected_time[
                (ticket, employee)
            ]
        )

        fallback_reason = (
            "Empleado más rápido sin coincidencia de equipo"
        )

    eligibility[
        (ticket, fallback_employee)
    ] = True

    fallback_records.append({
        "ticket": ticket,
        "employee": fallback_employee,
        "reason": fallback_reason,
        "predicted_hours": expected_time[
            (ticket, fallback_employee)
        ]
    })


# Actualizar el DataFrame con los respaldos aplicados
for record in fallback_records:

    condition = (
        (eligibility_df["ticket"] == record["ticket"])
        &
        (
            eligibility_df["employee"]
            == record["employee"]
        )
    )

    eligibility_df.loc[
        condition,
        "eligible"
    ] = True

    eligibility_df.loc[
        condition,
        "fallback_applied"
    ] = True


fallback_df = pd.DataFrame(
    fallback_records
)


# ============================================================
# LISTA DE EMPLEADOS ELEGIBLES PARA CADA TICKET
# ============================================================

eligible_employees_by_ticket = {
    ticket: [
        employee
        for employee in employees
        if eligibility[(ticket, employee)]
    ]
    for ticket in ticket_ids
}


# ============================================================
# VALIDACIÓN DE LA MATRIZ DE ELEGIBILIDAD
# ============================================================

tickets_without_candidates = [
    ticket
    for ticket in ticket_ids
    if len(
        eligible_employees_by_ticket[ticket]
    ) == 0
]

if tickets_without_candidates:

    raise ValueError(
        "Existen tickets sin empleados elegibles: "
        + ", ".join(
            map(str, tickets_without_candidates)
        )
    )


# ============================================================
# RESUMEN DE RESULTADOS
# ============================================================

total_combinations = (
    len(ticket_ids)
    * len(employees)
)

eligible_combinations = sum(
    eligibility.values()
)

ineligible_combinations = (
    total_combinations
    - eligible_combinations
)

eligible_counts = pd.Series({
    ticket: len(
        eligible_employees_by_ticket[ticket]
    )
    for ticket in ticket_ids
})


print("\n------------------------------------------")
print("Matriz de elegibilidad construida")
print("------------------------------------------")

print(
    "Combinaciones totales:",
    total_combinations
)

print(
    "Combinaciones elegibles:",
    eligible_combinations
)

print(
    "Combinaciones excluidas:",
    ineligible_combinations
)

print(
    "Porcentaje de combinaciones elegibles:",
    f"{100 * eligible_combinations / total_combinations:.2f}%"
)

print(
    "Candidatos mínimos por ticket:",
    int(eligible_counts.min())
)

print(
    "Candidatos medios por ticket:",
    round(eligible_counts.mean(), 2)
)

print(
    "Candidatos máximos por ticket:",
    int(eligible_counts.max())
)

print(
    "Tickets con mecanismo de respaldo:",
    len(fallback_records)
)


# ------------------------------------------------------------
# Vista previa de la matriz
# ------------------------------------------------------------

eligibility_matrix_df = (
    eligibility_df
    .pivot(
        index="ticket",
        columns="employee",
        values="eligible"
    )
    .reindex(
        index=ticket_ids,
        columns=employees
    )
)

print("\nVista previa de la matriz de elegibilidad:")

display(
    eligibility_matrix_df
    .iloc[:10, :10]
    .astype(int)
)


# ------------------------------------------------------------
# Mostrar casos en los que se aplicó respaldo
# ------------------------------------------------------------

if not fallback_df.empty:

    print(
        "\nTickets para los que se aplicó "
        "el mecanismo de respaldo:"
    )

    display(
        fallback_df.head(20)
    )

else:

    print(
        "\nNo fue necesario aplicar "
        "el mecanismo de respaldo."
    )

Construyendo matriz de elegibilidad...

------------------------------------------
Matriz de elegibilidad construida
------------------------------------------
Combinaciones totales: 8000
Combinaciones elegibles: 1995
Combinaciones excluidas: 6005
Porcentaje de combinaciones elegibles: 24.94%
Candidatos mínimos por ticket: 1
Candidatos medios por ticket: 9.98
Candidatos máximos por ticket: 18
Tickets con mecanismo de respaldo: 15

Vista previa de la matriz de elegibilidad:


employee,Andre Velicev,Andy Van Puyvelde,Anthony Restiau,Anthony Roels,Ayoub Azzouz,Bart Jansen,Bjorn Van Cotthem,Camila Reis,Carolina Vaeza,Christophe Kokken
ticket,,,,,,,,,,
DIG-23000,0,0,0,0,0,0,0,0,0,0
DIG-11893,0,1,0,0,0,1,0,0,1,0
DIG-20407,0,0,1,0,0,0,0,1,0,1
DIG-16326,0,0,0,0,0,0,0,0,0,0
DIG-21635,0,0,1,0,0,0,0,1,0,1
DIG-4349,0,0,1,0,0,0,0,1,0,1
DIG-18650,0,0,1,0,0,0,0,1,0,1
DIG-18391,0,0,1,0,0,0,0,1,0,1
DIG-14124,0,0,1,0,0,0,0,1,0,1



Tickets para los que se aplicó el mecanismo de respaldo:


,ticket,employee,reason,predicted_hours
0,DIG-3881,Elias Steenackers,Empleado más rápido del mismo equipo,88.293442
1,DIG-3518,Elias Steenackers,Empleado más rápido del mismo equipo,88.293442
2,DIG-2535,Elias Steenackers,Empleado más rápido del mismo equipo,88.293442
3,DIG-3109,Elias Steenackers,Empleado más rápido del mismo equipo,88.293442
4,DIG-23061,Elias Steenackers,Empleado más rápido del mismo equipo,88.293442
5,DIG-11964,Elias Steenackers,Empleado más rápido del mismo equipo,88.293442
6,DIG-19564,Bjorn Van Cotthem,Empleado más rápido del mismo equipo,24.926083
7,DIG-16611,Kobe Lenjou,Empleado más rápido del mismo equipo,25.327838
8,DIG-11549,Elias Steenackers,Empleado más rápido del mismo equipo,88.293442
9,DIG-2515,Elias Steenackers,Empleado más rápido del mismo equipo,88.293442


## 10. Validación de la instancia de optimización

Antes de formular el modelo COP, se realiza una validación de la instancia construida. El objetivo de esta etapa es comprobar que los datos y parámetros necesarios para resolver el problema son coherentes y que no existen errores que puedan provocar que el modelo resulte inviable o genere asignaciones incorrectas.

La validación incluye la comprobación de que cada ticket dispone de al menos un empleado elegible, que todos los tiempos esperados son valores numéricos positivos y que las capacidades de los empleados también son válidas. Asimismo, se analiza si la capacidad total disponible es suficiente para absorber la carga de trabajo mínima estimada de los tickets.

Esta comprobación previa no garantiza por sí sola la existencia de una solución factible, ya que las restricciones de equipo, prioridad y capacidad pueden interactuar de forma más compleja. Sin embargo, permite detectar anticipadamente los principales problemas de calidad o consistencia de la instancia antes de ejecutar el solver.

In [10]:
# ============================================================
# VALIDACIÓN DE LA INSTANCIA DE OPTIMIZACIÓN
# ============================================================

print("Validando la instancia de optimización...")

validation_errors = []
validation_warnings = []


# ------------------------------------------------------------
# 1. Validación de los conjuntos principales
# ------------------------------------------------------------

if len(ticket_ids) == 0:
    validation_errors.append(
        "La instancia no contiene ningún ticket."
    )

if len(employees) == 0:
    validation_errors.append(
        "La instancia no contiene ningún empleado."
    )

if len(ticket_ids) != len(set(ticket_ids)):
    validation_errors.append(
        "Existen identificadores de ticket duplicados."
    )

if len(employees) != len(set(employees)):
    validation_errors.append(
        "Existen empleados duplicados en la lista."
    )


# ------------------------------------------------------------
# 2. Validación de la matriz de tiempos esperados
# ------------------------------------------------------------

missing_time_values = []
invalid_time_values = []

for ticket in ticket_ids:

    for employee in employees:

        key = (ticket, employee)

        if key not in expected_time:

            missing_time_values.append(key)
            continue

        value = expected_time[key]

        if (
            value is None
            or not np.isfinite(value)
            or value <= 0
        ):

            invalid_time_values.append(
                (ticket, employee, value)
            )


if missing_time_values:

    validation_errors.append(
        f"Faltan {len(missing_time_values)} valores "
        "en la matriz de tiempos esperados."
    )


if invalid_time_values:

    validation_errors.append(
        f"Existen {len(invalid_time_values)} tiempos "
        "esperados inválidos, nulos o no positivos."
    )


# ------------------------------------------------------------
# 3. Validación de la capacidad de los empleados
# ------------------------------------------------------------

invalid_capacities = []

for employee in employees:

    capacity = employee_capacity.get(employee)

    if (
        capacity is None
        or not np.isfinite(capacity)
        or capacity <= 0
    ):

        invalid_capacities.append(
            (employee, capacity)
        )


if invalid_capacities:

    validation_errors.append(
        f"Existen {len(invalid_capacities)} empleados "
        "con una capacidad inválida."
    )


# ------------------------------------------------------------
# 4. Validación de empleados elegibles por ticket
# ------------------------------------------------------------

tickets_without_eligible_employees = []

for ticket in ticket_ids:

    candidates = eligible_employees_by_ticket.get(
        ticket,
        []
    )

    if len(candidates) == 0:

        tickets_without_eligible_employees.append(
            ticket
        )


if tickets_without_eligible_employees:

    validation_errors.append(
        f"Existen {len(tickets_without_eligible_employees)} "
        "tickets sin empleados elegibles."
    )


# ------------------------------------------------------------
# 5. Validación de prioridades y equipos
# ------------------------------------------------------------

tickets_without_priority = [
    ticket
    for ticket in ticket_ids
    if (
        ticket not in ticket_priority
        or pd.isna(ticket_priority[ticket])
        or str(ticket_priority[ticket]).strip() == ""
    )
]

if tickets_without_priority:

    validation_errors.append(
        f"Existen {len(tickets_without_priority)} "
        "tickets sin prioridad."
    )


tickets_without_team = [
    ticket
    for ticket in ticket_ids
    if (
        ticket not in ticket_team
        or pd.isna(ticket_team[ticket])
        or str(ticket_team[ticket]).strip() == ""
    )
]

if tickets_without_team:

    validation_warnings.append(
        f"Existen {len(tickets_without_team)} "
        "tickets sin un equipo claramente definido."
    )


employees_without_team = [
    employee
    for employee in employees
    if (
        employee not in employee_team
        or pd.isna(employee_team[employee])
        or str(employee_team[employee]).strip() == ""
    )
]

if employees_without_team:

    validation_warnings.append(
        f"Existen {len(employees_without_team)} "
        "empleados sin un equipo claramente definido."
    )


# ------------------------------------------------------------
# 6. Carga mínima estimada de cada ticket
# ------------------------------------------------------------

minimum_time_by_ticket = {}

for ticket in ticket_ids:

    eligible_candidates = (
        eligible_employees_by_ticket[ticket]
    )

    if len(eligible_candidates) > 0:

        minimum_time_by_ticket[ticket] = min(
            expected_time[(ticket, employee)]
            for employee in eligible_candidates
        )


minimum_required_capacity = sum(
    minimum_time_by_ticket.values()
)

total_available_capacity = sum(
    employee_capacity[employee]
    for employee in employees
)


# La capacidad insuficiente se considera una advertencia,
# no un error estructural. El solver comprobará formalmente
# la factibilidad de la instancia.

if minimum_required_capacity > total_available_capacity:

    validation_warnings.append(
        "La capacidad total disponible es inferior "
        "al tiempo mínimo estimado necesario para resolver "
        "todos los tickets. Es posible que el modelo COP "
        "resulte inviable."
    )

elif minimum_required_capacity > 0.90 * total_available_capacity:

    validation_warnings.append(
        "La carga mínima estimada supera el 90 % "
        "de la capacidad total disponible."
    )


# ------------------------------------------------------------
# 7. Capacidad disponible por equipo
# ------------------------------------------------------------

team_capacity = {}

for employee in employees:

    team = employee_team.get(
        employee,
        "SIN_EQUIPO"
    )

    team_capacity[team] = (
        team_capacity.get(team, 0)
        + employee_capacity[employee]
    )


minimum_required_by_team = {}

for ticket in ticket_ids:

    team = ticket_team[ticket]

    minimum_required_by_team[team] = (
        minimum_required_by_team.get(team, 0)
        + minimum_time_by_ticket.get(ticket, 0)
    )


team_capacity_analysis = []

for team, required_capacity in minimum_required_by_team.items():

    available_capacity = team_capacity.get(
        team,
        0
    )

    capacity_difference = (
        available_capacity
        - required_capacity
    )

    if available_capacity > 0:

        minimum_utilization = (
            required_capacity
            / available_capacity
        ) * 100

    else:

        minimum_utilization = np.inf

    team_capacity_analysis.append({
        "team": team,
        "minimum_required_hours": required_capacity,
        "available_capacity_hours": available_capacity,
        "capacity_difference_hours": capacity_difference,
        "minimum_utilization_percentage": minimum_utilization
    })

    if (
        ENFORCE_TEAM
        and required_capacity > available_capacity
    ):

        validation_warnings.append(
            f"El equipo '{team}' presenta una carga mínima "
            f"estimada de {required_capacity:.2f} horas, "
            f"superior a su capacidad disponible de "
            f"{available_capacity:.2f} horas."
        )


team_capacity_df = pd.DataFrame(
    team_capacity_analysis
)


# ------------------------------------------------------------
# 8. Resumen numérico de la instancia
# ------------------------------------------------------------

number_of_variables = sum(
    len(
        eligible_employees_by_ticket[ticket]
    )
    for ticket in ticket_ids
)

total_possible_variables = (
    len(ticket_ids)
    * len(employees)
)


if total_possible_variables > 0:

    variable_reduction = (
        1
        - number_of_variables
        / total_possible_variables
    ) * 100

else:

    variable_reduction = 0


if total_available_capacity > 0:

    minimum_utilization_percentage = (
        minimum_required_capacity
        / total_available_capacity
    ) * 100

else:

    minimum_utilization_percentage = np.inf


validation_summary = pd.DataFrame({
    "Indicador": [
        "Número de tickets",
        "Número de empleados",
        "Combinaciones posibles",
        "Combinaciones elegibles",
        "Reducción del espacio de variables",
        "Capacidad total disponible",
        "Carga mínima estimada",
        "Porcentaje mínimo de utilización",
        "Tickets con respaldo",
        "Errores estructurales",
        "Advertencias"
    ],
    "Valor": [
        len(ticket_ids),
        len(employees),
        total_possible_variables,
        number_of_variables,
        f"{variable_reduction:.2f} %",
        f"{total_available_capacity:.2f} horas",
        f"{minimum_required_capacity:.2f} horas",
        (
            f"{minimum_utilization_percentage:.2f} %"
            if np.isfinite(minimum_utilization_percentage)
            else "No disponible"
        ),
        len(fallback_records),
        len(validation_errors),
        len(validation_warnings)
    ]
})


# ------------------------------------------------------------
# 9. Presentación del resultado
# ------------------------------------------------------------

print("\n------------------------------------------")
print("RESUMEN DE VALIDACIÓN")
print("------------------------------------------")

display(validation_summary)


if not team_capacity_df.empty:

    print("\nCapacidad mínima estimada por equipo:")

    team_capacity_display = (
        team_capacity_df.copy()
    )

    team_capacity_display[
        "minimum_required_hours"
    ] = (
        team_capacity_display[
            "minimum_required_hours"
        ]
        .round(2)
    )

    team_capacity_display[
        "available_capacity_hours"
    ] = (
        team_capacity_display[
            "available_capacity_hours"
        ]
        .round(2)
    )

    team_capacity_display[
        "capacity_difference_hours"
    ] = (
        team_capacity_display[
            "capacity_difference_hours"
        ]
        .round(2)
    )

    team_capacity_display[
        "minimum_utilization_percentage"
    ] = (
        team_capacity_display[
            "minimum_utilization_percentage"
        ]
        .replace([np.inf, -np.inf], np.nan)
        .round(2)
    )

    display(
        team_capacity_display
        .sort_values(
            by="minimum_utilization_percentage",
            ascending=False,
            na_position="first"
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 10. Mostrar advertencias
# ------------------------------------------------------------

if validation_warnings:

    print("\nAdvertencias detectadas:")

    for warning in validation_warnings:
        print("⚠", warning)

else:

    print("\nNo se detectaron advertencias.")


# ------------------------------------------------------------
# 11. Tratamiento de errores estructurales
# ------------------------------------------------------------

if validation_errors:

    print("\nErrores estructurales detectados:")

    for error in validation_errors:
        print("✖", error)

    raise ValueError(
        "La instancia contiene errores estructurales "
        "que impiden formular correctamente el modelo COP. "
        "Deben corregirse los errores anteriores."
    )


# ------------------------------------------------------------
# 12. Resultado final de la validación
# ------------------------------------------------------------

if validation_warnings:

    print(
        "\n✓ La estructura de la instancia es válida."
    )

    print(
        "⚠ Se continuará con la formulación del modelo, "
        "aunque existen advertencias relacionadas con "
        "la capacidad o la posible factibilidad."
    )

    print(
        "El solver determinará formalmente si existe "
        "una solución que respete todas las restricciones."
    )

else:

    print(
        "\n✓ La instancia ha superado correctamente "
        "todas las comprobaciones de validación."
    )

Validando la instancia de optimización...

------------------------------------------
RESUMEN DE VALIDACIÓN
------------------------------------------


,Indicador,Valor
0,Número de tickets,200
1,Número de empleados,40
2,Combinaciones posibles,8000
3,Combinaciones elegibles,1995
4,Reducción del espacio de variables,75.06 %
5,Capacidad total disponible,1600.00 horas
6,Carga mínima estimada,1633.89 horas
7,Porcentaje mínimo de utilización,102.12 %
8,Tickets con respaldo,15
9,Errores estructurales,0



Capacidad mínima estimada por equipo:


,team,minimum_required_hours,available_capacity_hours,capacity_difference_hours,minimum_utilization_percentage
0,DW,240.84,40.0,-200.84,602.10
1,CS,1068.93,720.0,-348.93,148.46
2,IT,271.32,320.0,48.68,84.79
3,DATA,38.10,200.0,161.90,19.05
4,BCFIN,14.69,200.0,185.31,7.35



Advertencias detectadas:
⚠ La capacidad total disponible es inferior al tiempo mínimo estimado necesario para resolver todos los tickets. Es posible que el modelo COP resulte inviable.
⚠ El equipo 'CS' presenta una carga mínima estimada de 1068.93 horas, superior a su capacidad disponible de 720.00 horas.
⚠ El equipo 'DW' presenta una carga mínima estimada de 240.84 horas, superior a su capacidad disponible de 40.00 horas.

✓ La estructura de la instancia es válida.
⚠ Se continuará con la formulación del modelo, aunque existen advertencias relacionadas con la capacidad o la posible factibilidad.
El solver determinará formalmente si existe una solución que respete todas las restricciones.


## 11. Formulación del modelo COP

Una vez construidos y validados los parámetros de la instancia, se procede a formular el problema de optimización mediante Constraint Optimization Programming. La implementación se realiza utilizando el solver CP-SAT de Google OR-Tools, especialmente diseñado para resolver problemas combinatorios con variables enteras y binarias.

Para cada combinación elegible entre un ticket \(i\) y un empleado \(j\), se define una variable binaria \(\delta_{ij}\). Esta variable toma el valor uno cuando el ticket es asignado al empleado y cero en caso contrario. Las combinaciones consideradas no elegibles durante la etapa anterior no generan una variable de decisión, reduciendo así el tamaño del modelo.

El modelo incorpora la restricción de asignación única, según la cual cada ticket debe ser asignado exactamente a un empleado. Asimismo, se limita la carga total asignada a cada trabajador para que no supere su capacidad disponible. Las restricciones de equipo, experiencia por prioridad y cumplimiento del SLA quedan incorporadas mediante la matriz de elegibilidad construida previamente.

La función objetivo minimiza el tiempo total esperado de resolución y, simultáneamente, penaliza las diferencias entre la carga asignada a cada empleado y la carga media del conjunto. Debido a que CP-SAT trabaja con valores enteros, los tiempos expresados en horas se transforman mediante un factor de escala que permite conservar dos posiciones decimales.

In [11]:
# ============================================================
# 11. FORMULACIÓN DEL MODELO COP
# ============================================================

print("Formulando el modelo COP...")


# ------------------------------------------------------------
# 11.1. Comprobación de objetos necesarios
# ------------------------------------------------------------

required_objects = [
    "ticket_ids",
    "employees",
    "expected_time",
    "employee_capacity",
    "eligible_employees_by_ticket",
    "BETA_BALANCE",
    "TIME_SCALE"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "No se han creado los siguientes objetos necesarios:\n\n"
        + "\n".join(missing_objects)
        + "\n\nEjecuta nuevamente las secciones anteriores."
    )


# Comprobaciones adicionales

if len(ticket_ids) == 0:
    raise ValueError(
        "No existen tickets en la instancia."
    )

if len(employees) == 0:
    raise ValueError(
        "No existen empleados en la instancia."
    )

if TIME_SCALE <= 0:
    raise ValueError(
        "TIME_SCALE debe ser un número positivo."
    )


# ------------------------------------------------------------
# 11.2. Creación del modelo
# ------------------------------------------------------------

# Se crea un modelo nuevo cada vez que se ejecuta esta celda.
# Esto evita conservar variables o restricciones de ejecuciones
# anteriores.

model = cp_model.CpModel()


# ------------------------------------------------------------
# 11.3. Índices auxiliares
# ------------------------------------------------------------

# Se crean índices numéricos para utilizar nombres simples
# y seguros en las variables internas de OR-Tools.

ticket_index = {
    ticket: i
    for i, ticket in enumerate(ticket_ids)
}

employee_index = {
    employee: j
    for j, employee in enumerate(employees)
}


# ------------------------------------------------------------
# 11.4. Conversión de tiempos y capacidades a enteros
# ------------------------------------------------------------

# CP-SAT trabaja con coeficientes enteros.
# TIME_SCALE = 100 conserva dos posiciones decimales.

scaled_expected_time = {}

for ticket in ticket_ids:

    for employee in employees:

        value = expected_time[(ticket, employee)]

        if (
            value is None
            or not np.isfinite(value)
            or value <= 0
        ):
            raise ValueError(
                f"Tiempo esperado inválido para "
                f"({ticket}, {employee}): {value}"
            )

        scaled_expected_time[(ticket, employee)] = max(
            1,
            int(round(value * TIME_SCALE))
        )


scaled_employee_capacity = {}

for employee in employees:

    capacity = employee_capacity[employee]

    if (
        capacity is None
        or not np.isfinite(capacity)
        or capacity <= 0
    ):
        raise ValueError(
            f"Capacidad inválida para {employee}: "
            f"{capacity}"
        )

    scaled_employee_capacity[employee] = int(
        round(capacity * TIME_SCALE)
    )


# ------------------------------------------------------------
# 11.5. Variables binarias de asignación δ_ij
# ------------------------------------------------------------

assignment_variables = {}

for ticket in ticket_ids:

    candidates = eligible_employees_by_ticket.get(
        ticket,
        []
    )

    if len(candidates) == 0:
        raise ValueError(
            f"El ticket {ticket} no tiene empleados elegibles."
        )

    for employee in candidates:

        i = ticket_index[ticket]
        j = employee_index[employee]

        assignment_variables[(ticket, employee)] = (
            model.NewBoolVar(
                f"delta_{i}_{j}"
            )
        )


print(
    "Variables binarias creadas:",
    len(assignment_variables)
)


# ------------------------------------------------------------
# 11.6. Restricción de asignación única
# ------------------------------------------------------------

# Cada ticket debe asignarse exactamente a un empleado.

for ticket in ticket_ids:

    eligible_variables = [
        assignment_variables[(ticket, employee)]
        for employee
        in eligible_employees_by_ticket[ticket]
    ]

    model.Add(
        sum(eligible_variables) == 1
    )


# ------------------------------------------------------------
# 11.7. Carga y capacidad máxima por empleado
# ------------------------------------------------------------

employee_load_variables = {}

for employee in employees:

    j = employee_index[employee]
    capacity = scaled_employee_capacity[employee]

    # La carga del empleado se mide en horas escaladas.
    employee_load_variables[employee] = (
        model.NewIntVar(
            0,
            capacity,
            f"load_{j}"
        )
    )

    assigned_time_terms = [
        scaled_expected_time[(ticket, employee)]
        * assignment_variables[(ticket, employee)]

        for ticket in ticket_ids

        if (ticket, employee)
        in assignment_variables
    ]

    if assigned_time_terms:

        model.Add(
            employee_load_variables[employee]
            == sum(assigned_time_terms)
        )

    else:

        model.Add(
            employee_load_variables[employee] == 0
        )

    # La carga total no puede superar la capacidad.
    model.Add(
        employee_load_variables[employee]
        <= capacity
    )


# ------------------------------------------------------------
# 11.8. Carga total del sistema
# ------------------------------------------------------------

maximum_total_capacity = sum(
    scaled_employee_capacity.values()
)

total_load_variable = model.NewIntVar(
    0,
    maximum_total_capacity,
    "total_load"
)

model.Add(
    total_load_variable
    == sum(
        employee_load_variables[employee]
        for employee in employees
    )
)


# ------------------------------------------------------------
# 11.9. Desviación respecto a la carga media
# ------------------------------------------------------------

# La formulación original contiene:
#
# |L_j - L_media|
#
# Como la media requiere una división, se utiliza:
#
# |m * L_j - carga_total|
#
# donde m es el número de empleados.
#
# Esta expresión equivale a:
#
# m * |L_j - L_media|

number_of_employees = len(employees)

maximum_employee_capacity = max(
    scaled_employee_capacity.values()
)

# Corrección: se utiliza max(), no maximum().
maximum_deviation = (
    number_of_employees
    * maximum_employee_capacity
    + maximum_total_capacity
)

absolute_deviation_variables = {}

for employee in employees:

    j = employee_index[employee]

    signed_deviation = model.NewIntVar(
        -maximum_deviation,
        maximum_deviation,
        f"signed_deviation_{j}"
    )

    absolute_deviation = model.NewIntVar(
        0,
        maximum_deviation,
        f"absolute_deviation_{j}"
    )

    model.Add(
        signed_deviation
        ==
        number_of_employees
        * employee_load_variables[employee]
        - total_load_variable
    )

    model.AddAbsEquality(
        absolute_deviation,
        signed_deviation
    )

    absolute_deviation_variables[employee] = (
        absolute_deviation
    )


# ------------------------------------------------------------
# 11.10. Función objetivo
# ------------------------------------------------------------

# Función objetivo original:
#
# minimizar:
#
# suma(t_ij * delta_ij)
# +
# beta * suma(|L_j - L_media|)
#
# Como la desviación se ha multiplicado por el número de
# empleados, también se multiplica el término temporal por
# ese mismo número. De esta forma se conserva la relación
# entre ambos componentes de la función objetivo.

OBJECTIVE_SCALE = 1000

time_weight = (
    OBJECTIVE_SCALE
    * number_of_employees
)

balance_weight = int(
    round(
        BETA_BALANCE
        * OBJECTIVE_SCALE
    )
)


total_expected_time_term = sum(
    scaled_expected_time[(ticket, employee)]
    * assignment_variables[(ticket, employee)]

    for ticket, employee
    in assignment_variables.keys()
)


total_balance_term = sum(
    absolute_deviation_variables[employee]
    for employee in employees
)


model.Minimize(
    time_weight
    * total_expected_time_term
    +
    balance_weight
    * total_balance_term
)


# ------------------------------------------------------------
# 11.11. Resumen del modelo formulado
# ------------------------------------------------------------

number_of_assignment_constraints = len(
    ticket_ids
)

number_of_capacity_constraints = len(
    employees
)

number_of_balance_variables = len(
    absolute_deviation_variables
)

total_possible_assignments = (
    len(ticket_ids)
    * len(employees)
)

variable_reduction_percentage = (
    100
    * (
        1
        - len(assignment_variables)
        / total_possible_assignments
    )
    if total_possible_assignments > 0
    else 0
)


model_summary = pd.DataFrame({
    "Elemento": [
        "Tickets",
        "Empleados",
        "Combinaciones posibles",
        "Variables binarias creadas",
        "Reducción de variables",
        "Restricciones de asignación única",
        "Restricciones de capacidad",
        "Variables de carga",
        "Variables de desviación absoluta",
        "Peso del tiempo total",
        "Peso del balance",
        "Escala temporal"
    ],
    "Valor": [
        len(ticket_ids),
        len(employees),
        total_possible_assignments,
        len(assignment_variables),
        f"{variable_reduction_percentage:.2f} %",
        number_of_assignment_constraints,
        number_of_capacity_constraints,
        len(employee_load_variables),
        number_of_balance_variables,
        time_weight,
        balance_weight,
        TIME_SCALE
    ]
})


print("\n------------------------------------------")
print("MODELO COP FORMULADO")
print("------------------------------------------")

display(model_summary)

print(
    "\n✓ La formulación matemática se ha "
    "construido correctamente."
)

print(
    "La siguiente sección ejecutará el solver "
    "para determinar si existe una solución factible."
)

Formulando el modelo COP...
Variables binarias creadas: 1995

------------------------------------------
MODELO COP FORMULADO
------------------------------------------


,Elemento,Valor
0,Tickets,200
1,Empleados,40
2,Combinaciones posibles,8000
3,Variables binarias creadas,1995
4,Reducción de variables,75.06 %
5,Restricciones de asignación única,200
6,Restricciones de capacidad,40
7,Variables de carga,40
8,Variables de desviación absoluta,40
9,Peso del tiempo total,40000



✓ La formulación matemática se ha construido correctamente.
La siguiente sección ejecutará el solver para determinar si existe una solución factible.


## 12. Resolución del modelo

Una vez formulado el problema, se configura y ejecuta el solver CP-SAT de Google OR-Tools. El solver explorará el espacio de soluciones factibles con el objetivo de minimizar la función objetivo definida en la sección anterior.

El proceso finalizará cuando se demuestre la optimalidad de la solución, cuando se alcance el límite máximo de tiempo configurado o cuando se determine que la instancia es inviable. El estado devuelto por el solver permite distinguir entre una solución óptima, una solución factible cuya optimalidad no ha podido demostrarse y una instancia sin solución.

Para favorecer la reproducibilidad del experimento, se establece una semilla aleatoria fija y se registran tanto el tiempo real de ejecución como las principales estadísticas internas del proceso de búsqueda.

In [12]:
# ============================================================
# 12. RESOLUCIÓN DEL MODELO
# ============================================================

print("Ejecutando el solver CP-SAT...")


# ------------------------------------------------------------
# 12.1. Comprobación de objetos necesarios
# ------------------------------------------------------------

required_solver_objects = [
    "model",
    "assignment_variables",
    "employee_load_variables",
    "total_load_variable",
    "MAX_SOLVER_TIME_SECONDS",
    "NUM_SEARCH_WORKERS",
    "RANDOM_SEED",
    "ticket_ids",
    "employees"
]

missing_solver_objects = [
    object_name
    for object_name in required_solver_objects
    if object_name not in globals()
]

if missing_solver_objects:
    raise RuntimeError(
        "No se han creado los siguientes objetos necesarios:\n\n"
        + "\n".join(missing_solver_objects)
        + "\n\nEjecuta nuevamente la sección 11."
    )


# ------------------------------------------------------------
# 12.2. Configuración del solver
# ------------------------------------------------------------

solver = cp_model.CpSolver()

# Tiempo máximo permitido para la búsqueda.
solver.parameters.max_time_in_seconds = float(
    MAX_SOLVER_TIME_SECONDS
)

# Número de procesadores utilizados.
solver.parameters.num_search_workers = int(
    NUM_SEARCH_WORKERS
)

# Semilla para mejorar la reproducibilidad.
solver.parameters.random_seed = int(
    RANDOM_SEED
)

# Mostrar información detallada del solver.
# Cambia a True si deseas ver el progreso interno.
solver.parameters.log_search_progress = False


# ------------------------------------------------------------
# 12.3. Ejecución del modelo
# ------------------------------------------------------------

start_solver_time = perf_counter()

solver_status = solver.Solve(model)

solver_runtime = (
    perf_counter()
    - start_solver_time
)

solver_status_name = solver.StatusName(
    solver_status
)


# ------------------------------------------------------------
# 12.4. Interpretación del estado
# ------------------------------------------------------------

if solver_status == cp_model.OPTIMAL:

    solution_message = (
        "Se encontró una solución óptima y se demostró "
        "que no existe ninguna asignación mejor."
    )

    solution_exists = True
    proven_optimal = True


elif solver_status == cp_model.FEASIBLE:

    solution_message = (
        "Se encontró una solución factible, pero no fue "
        "posible demostrar su optimalidad dentro del "
        "límite temporal establecido."
    )

    solution_exists = True
    proven_optimal = False


elif solver_status == cp_model.INFEASIBLE:

    solution_message = (
        "La instancia es inviable. No existe ninguna "
        "asignación que cumpla simultáneamente todas "
        "las restricciones del modelo."
    )

    solution_exists = False
    proven_optimal = False


elif solver_status == cp_model.MODEL_INVALID:

    solution_message = (
        "El modelo es inválido. Debe revisarse la "
        "formulación matemática o los límites de las "
        "variables y restricciones."
    )

    solution_exists = False
    proven_optimal = False


else:

    solution_message = (
        "El solver no pudo determinar una solución "
        "dentro de las condiciones establecidas."
    )

    solution_exists = False
    proven_optimal = False


# ------------------------------------------------------------
# 12.5. Estadísticas del proceso de resolución
# ------------------------------------------------------------

solver_summary = pd.DataFrame({
    "Indicador": [
        "Estado del solver",
        "Existe solución",
        "Optimalidad demostrada",
        "Tiempo máximo configurado",
        "Tiempo real de ejecución",
        "Número de tickets",
        "Número de empleados",
        "Variables binarias",
        "Conflictos explorados",
        "Ramificaciones exploradas",
        "Tiempo interno del solver"
    ],
    "Valor": [
        solver_status_name,
        solution_exists,
        proven_optimal,
        f"{MAX_SOLVER_TIME_SECONDS:.2f} segundos",
        f"{solver_runtime:.4f} segundos",
        len(ticket_ids),
        len(employees),
        len(assignment_variables),
        solver.NumConflicts(),
        solver.NumBranches(),
        f"{solver.WallTime():.4f} segundos"
    ]
})


# ------------------------------------------------------------
# 12.6. Valor de la función objetivo y cota
# ------------------------------------------------------------

if solution_exists:

    solver_objective_value = solver.ObjectiveValue()

    solver_best_bound = solver.BestObjectiveBound()

    if abs(solver_objective_value) > 1e-12:

        solver_relative_gap = (
            abs(
                solver_objective_value
                - solver_best_bound
            )
            / abs(solver_objective_value)
        ) * 100

    else:

        solver_relative_gap = 0.0

else:

    solver_objective_value = np.nan
    solver_best_bound = np.nan
    solver_relative_gap = np.nan


objective_summary = pd.DataFrame({
    "Indicador": [
        "Valor interno de la función objetivo",
        "Mejor cota encontrada",
        "Gap relativo"
    ],
    "Valor": [
        (
            f"{solver_objective_value:.2f}"
            if np.isfinite(solver_objective_value)
            else "No disponible"
        ),
        (
            f"{solver_best_bound:.2f}"
            if np.isfinite(solver_best_bound)
            else "No disponible"
        ),
        (
            f"{solver_relative_gap:.6f} %"
            if np.isfinite(solver_relative_gap)
            else "No disponible"
        )
    ]
})


# ------------------------------------------------------------
# 12.7. Presentación de resultados
# ------------------------------------------------------------

print("\n------------------------------------------")
print("RESULTADO DEL SOLVER")
print("------------------------------------------")

display(solver_summary)

print("\nEvaluación de la función objetivo:")

display(objective_summary)

print("\nInterpretación:")

print(solution_message)


# ------------------------------------------------------------
# 12.8. Mensaje final
# ------------------------------------------------------------

if solver_status == cp_model.OPTIMAL:

    print(
        "\n✓ El modelo ha sido resuelto de forma óptima."
    )

elif solver_status == cp_model.FEASIBLE:

    print(
        "\n✓ Se dispone de una solución factible."
    )

    print(
        "⚠ La optimalidad no ha sido demostrada."
    )

elif solver_status == cp_model.INFEASIBLE:

    print(
        "\n⚠ No existe una solución factible con la "
        "configuración actual."
    )

    print(
        "Para obtener una solución será necesario reducir "
        "el número de tickets, aumentar la capacidad o "
        "flexibilizar alguna de las restricciones."
    )

elif solver_status == cp_model.MODEL_INVALID:

    print(
        "\n✖ El solver ha identificado un modelo inválido."
    )

else:

    print(
        "\n⚠ El solver finalizó sin encontrar ni demostrar "
        "una solución."
    )

Ejecutando el solver CP-SAT...

------------------------------------------
RESULTADO DEL SOLVER
------------------------------------------


,Indicador,Valor
0,Estado del solver,INFEASIBLE
1,Existe solución,False
2,Optimalidad demostrada,False
3,Tiempo máximo configurado,300.00 segundos
4,Tiempo real de ejecución,0.0058 segundos
5,Número de tickets,200
6,Número de empleados,40
7,Variables binarias,1995
8,Conflictos explorados,0
9,Ramificaciones exploradas,0



Evaluación de la función objetivo:


,Indicador,Valor
0,Valor interno de la función objetivo,No disponible
1,Mejor cota encontrada,No disponible
2,Gap relativo,No disponible



Interpretación:
La instancia es inviable. No existe ninguna asignación que cumpla simultáneamente todas las restricciones del modelo.

⚠ No existe una solución factible con la configuración actual.
Para obtener una solución será necesario reducir el número de tickets, aumentar la capacidad o flexibilizar alguna de las restricciones.


## 13. Procesamiento y análisis de la solución

Cuando el solver encuentra una solución óptima o factible, se recuperan los valores de las variables binarias de asignación para identificar qué empleado ha sido seleccionado para cada ticket.

A partir de estas asignaciones se calculan los principales indicadores del resultado, incluyendo el tiempo esperado de resolución de cada ticket, la carga total asignada a cada empleado, el porcentaje de utilización de su capacidad, el cumplimiento del SLA y el equilibrio de la distribución del trabajo.

Los resultados se organizan en diferentes tablas para facilitar su interpretación. La primera tabla muestra la asignación detallada de cada ticket, mientras que la segunda resume la carga de trabajo por empleado. Finalmente, se genera un resumen general de la solución con los principales indicadores de rendimiento del modelo.

Cuando el solver determina que la instancia es inviable o no encuentra una solución dentro del límite temporal, esta sección no intenta recuperar asignaciones y genera únicamente un resumen del estado obtenido.

In [13]:
# ============================================================
# 13. PROCESAMIENTO Y ANÁLISIS DE LA SOLUCIÓN
# ============================================================

print("Procesando los resultados del modelo...")


# ------------------------------------------------------------
# 13.1. Comprobación de objetos necesarios
# ------------------------------------------------------------

required_result_objects = [
    "solver",
    "solver_status",
    "solution_exists",
    "proven_optimal",
    "assignment_variables",
    "expected_time",
    "employee_capacity",
    "ticket_ids",
    "employees",
    "ticket_priority",
    "ticket_team",
    "employee_team",
    "BETA_BALANCE",
    "solver_runtime"
]

missing_result_objects = [
    object_name
    for object_name in required_result_objects
    if object_name not in globals()
]

if missing_result_objects:

    raise RuntimeError(
        "No se han creado los siguientes objetos necesarios:\n\n"
        + "\n".join(missing_result_objects)
        + "\n\nEjecuta nuevamente las secciones 11 y 12."
    )


# ------------------------------------------------------------
# 13.2. Inicialización de tablas vacías
# ------------------------------------------------------------

assignments_df = pd.DataFrame()

employee_summary_df = pd.DataFrame()

solution_summary_df = pd.DataFrame()


# ------------------------------------------------------------
# 13.3. Recuperación de las asignaciones
# ------------------------------------------------------------

if solution_exists:

    assignment_records = []

    assigned_employee_by_ticket = {}

    for ticket in ticket_ids:

        selected_employee = None

        for employee in eligible_employees_by_ticket[ticket]:

            variable = assignment_variables.get(
                (ticket, employee)
            )

            if (
                variable is not None
                and solver.Value(variable) == 1
            ):

                selected_employee = employee
                break

        if selected_employee is None:

            raise RuntimeError(
                f"No fue posible recuperar el empleado "
                f"asignado al ticket {ticket}."
            )

        assigned_employee_by_ticket[ticket] = (
            selected_employee
        )


# ------------------------------------------------------------
# 13.4. Construcción de la tabla de asignaciones
# ------------------------------------------------------------

    for ticket in ticket_ids:

        employee = assigned_employee_by_ticket[ticket]

        predicted_hours = expected_time[
            (ticket, employee)
        ]

        priority = ticket_priority[ticket]

        sla_hours = float(
            SLA_BY_PRIORITY.get(
                str(priority).strip().upper(),
                np.inf
            )
        )

        sla_met = (
            predicted_hours <= sla_hours
            if np.isfinite(sla_hours)
            else True
        )

        assignment_records.append({
            "ticket": ticket,
            "priority": priority,
            "ticket_team": ticket_team[ticket],
            "assigned_employee": employee,
            "employee_team": employee_team.get(
                employee,
                "SIN_EQUIPO"
            ),
            "predicted_hours": predicted_hours,
            "sla_hours": (
                sla_hours
                if np.isfinite(sla_hours)
                else np.nan
            ),
            "sla_met": sla_met,
            "employee_capacity": employee_capacity[
                employee
            ],
            "fallback_applied": (
                any(
                    record["ticket"] == ticket
                    and record["employee"] == employee
                    for record in fallback_records
                )
            )
        })


    assignments_df = pd.DataFrame(
        assignment_records
    )


# ------------------------------------------------------------
# 13.5. Cálculo de la carga por empleado
# ------------------------------------------------------------

    employee_records = []

    for employee in employees:

        employee_assignments = assignments_df[
            assignments_df["assigned_employee"]
            == employee
        ]

        assigned_tickets = len(
            employee_assignments
        )

        assigned_load = float(
            employee_assignments[
                "predicted_hours"
            ].sum()
        )

        capacity = float(
            employee_capacity[employee]
        )

        utilization = (
            assigned_load / capacity
            if capacity > 0
            else np.nan
        )

        remaining_capacity = (
            capacity - assigned_load
        )

        employee_records.append({
            "employee": employee,
            "team": employee_team.get(
                employee,
                "SIN_EQUIPO"
            ),
            "assigned_tickets": assigned_tickets,
            "assigned_load_hours": assigned_load,
            "capacity_hours": capacity,
            "remaining_capacity_hours": (
                remaining_capacity
            ),
            "utilization_percentage": (
                utilization * 100
                if np.isfinite(utilization)
                else np.nan
            )
        })


    employee_summary_df = pd.DataFrame(
        employee_records
    )


# ------------------------------------------------------------
# 13.6. Cálculo de indicadores generales
# ------------------------------------------------------------

    total_processing_time = float(
        assignments_df[
            "predicted_hours"
        ].sum()
    )

    employee_loads = (
        employee_summary_df[
            "assigned_load_hours"
        ].to_numpy(dtype=float)
    )

    mean_employee_load = float(
        employee_loads.mean()
    )

    balance_deviation = float(
        np.abs(
            employee_loads
            - mean_employee_load
        ).sum()
    )

    real_objective_value = (
        total_processing_time
        + BETA_BALANCE
        * balance_deviation
    )

    sla_compliance_rate = float(
        assignments_df[
            "sla_met"
        ].mean() * 100
    )

    assigned_employees = int(
        (
            employee_summary_df[
                "assigned_tickets"
            ] > 0
        ).sum()
    )

    unused_employees = (
        len(employees)
        - assigned_employees
    )

    maximum_employee_load = float(
        employee_loads.max()
    )

    minimum_employee_load = float(
        employee_loads.min()
    )

    load_standard_deviation = float(
        employee_loads.std()
    )

    total_capacity = float(
        employee_summary_df[
            "capacity_hours"
        ].sum()
    )

    total_utilization = (
        total_processing_time
        / total_capacity
        * 100
        if total_capacity > 0
        else np.nan
    )


# ------------------------------------------------------------
# 13.7. Resumen general de la solución
# ------------------------------------------------------------

    solution_summary_df = pd.DataFrame([{
        "solver_status": solver.StatusName(
            solver_status
        ),
        "proven_optimal": proven_optimal,
        "n_tickets": len(ticket_ids),
        "n_employees": len(employees),
        "employees_with_assignments": (
            assigned_employees
        ),
        "employees_without_assignments": (
            unused_employees
        ),
        "total_processing_time_hours": (
            total_processing_time
        ),
        "mean_employee_load_hours": (
            mean_employee_load
        ),
        "maximum_employee_load_hours": (
            maximum_employee_load
        ),
        "minimum_employee_load_hours": (
            minimum_employee_load
        ),
        "load_standard_deviation": (
            load_standard_deviation
        ),
        "balance_deviation": (
            balance_deviation
        ),
        "beta_balance": BETA_BALANCE,
        "recalculated_objective": (
            real_objective_value
        ),
        "sla_compliance_percentage": (
            sla_compliance_rate
        ),
        "total_capacity_hours": (
            total_capacity
        ),
        "total_utilization_percentage": (
            total_utilization
        ),
        "solver_runtime_seconds": (
            solver_runtime
        )
    }])


# ------------------------------------------------------------
# 13.8. Formato de las tablas
# ------------------------------------------------------------

    assignments_display = assignments_df.copy()

    assignments_display[
        "predicted_hours"
    ] = (
        assignments_display[
            "predicted_hours"
        ].round(2)
    )

    assignments_display[
        "sla_hours"
    ] = (
        assignments_display[
            "sla_hours"
        ].round(2)
    )

    employee_summary_display = (
        employee_summary_df.copy()
    )

    numeric_employee_columns = [
        "assigned_load_hours",
        "capacity_hours",
        "remaining_capacity_hours",
        "utilization_percentage"
    ]

    for column in numeric_employee_columns:

        employee_summary_display[column] = (
            employee_summary_display[column]
            .round(2)
        )


# ------------------------------------------------------------
# 13.9. Presentación de resultados
# ------------------------------------------------------------

    print("\n------------------------------------------")
    print("ASIGNACIÓN DE TICKETS")
    print("------------------------------------------")

    display(
        assignments_display
        .sort_values(
            by=[
                "priority",
                "ticket"
            ]
        )
        .reset_index(drop=True)
    )


    print("\n------------------------------------------")
    print("CARGA DE TRABAJO POR EMPLEADO")
    print("------------------------------------------")

    display(
        employee_summary_display
        .sort_values(
            by="assigned_load_hours",
            ascending=False
        )
        .reset_index(drop=True)
    )


    print("\n------------------------------------------")
    print("RESUMEN GENERAL DE LA SOLUCIÓN")
    print("------------------------------------------")

    solution_summary_display = (
        solution_summary_df.copy()
    )

    numeric_summary_columns = [
        "total_processing_time_hours",
        "mean_employee_load_hours",
        "maximum_employee_load_hours",
        "minimum_employee_load_hours",
        "load_standard_deviation",
        "balance_deviation",
        "recalculated_objective",
        "sla_compliance_percentage",
        "total_capacity_hours",
        "total_utilization_percentage",
        "solver_runtime_seconds"
    ]

    for column in numeric_summary_columns:

        solution_summary_display[column] = (
            solution_summary_display[column]
            .round(4)
        )

    display(
        solution_summary_display
    )


# ------------------------------------------------------------
# 13.10. Comprobaciones finales
# ------------------------------------------------------------

    tickets_assigned = len(
        assignments_df
    )

    unique_tickets_assigned = (
        assignments_df[
            "ticket"
        ].nunique()
    )

    if tickets_assigned != len(ticket_ids):

        raise RuntimeError(
            "El número de asignaciones recuperadas "
            "no coincide con el número de tickets."
        )

    if unique_tickets_assigned != len(ticket_ids):

        raise RuntimeError(
            "Existen tickets duplicados o sin asignar "
            "en la solución recuperada."
        )

    capacity_violations = (
        employee_summary_df[
            employee_summary_df[
                "assigned_load_hours"
            ]
            >
            employee_summary_df[
                "capacity_hours"
            ]
            + 1e-6
        ]
    )

    if not capacity_violations.empty:

        raise RuntimeError(
            "La solución recuperada contiene empleados "
            "cuya carga supera la capacidad."
        )


    print(
        "\n✓ La solución ha sido procesada y "
        "validada correctamente."
    )


# ------------------------------------------------------------
# 13.11. Caso sin solución
# ------------------------------------------------------------

else:

    solution_summary_df = pd.DataFrame([{
        "solver_status": solver.StatusName(
            solver_status
        ),
        "proven_optimal": False,
        "n_tickets": len(ticket_ids),
        "n_employees": len(employees),
        "solution_available": False,
        "solver_runtime_seconds": (
            solver_runtime
        ),
        "interpretation": solution_message
    }])

    print("\n------------------------------------------")
    print("NO SE DISPONE DE UNA SOLUCIÓN")
    print("------------------------------------------")

    display(
        solution_summary_df
    )

    print(
        "\nNo se generan tablas de asignación "
        "ni de carga por empleado porque el solver "
        "no encontró una solución factible."
    )

    print(
        "Puedes reducir N_TICKETS, aumentar la "
        "capacidad disponible o flexibilizar alguna "
        "restricción y ejecutar nuevamente las "
        "secciones 7 a 13."
    )

Procesando los resultados del modelo...

------------------------------------------
NO SE DISPONE DE UNA SOLUCIÓN
------------------------------------------


,solver_status,proven_optimal,n_tickets,n_employees,solution_available,solver_runtime_seconds,interpretation
0,INFEASIBLE,False,200,40,False,0.005762,La instancia es inviable. No existe ninguna as...



No se generan tablas de asignación ni de carga por empleado porque el solver no encontró una solución factible.
Puedes reducir N_TICKETS, aumentar la capacidad disponible o flexibilizar alguna restricción y ejecutar nuevamente las secciones 7 a 13.


## 14. Exportación de resultados

En esta sección se exportan los resultados generados durante la resolución del modelo a la carpeta definida previamente en Google Drive. El objetivo es conservar tanto la solución final como las principales estructuras auxiliares empleadas durante la formulación del problema.

Cuando el solver encuentra una solución factible u óptima, se guardan la asignación detallada de tickets, la carga de trabajo por empleado y el resumen general del experimento. También se exportan la matriz de tiempos esperados, la matriz de elegibilidad y los resultados de la validación previa.

Si la instancia resulta inviable, se exporta igualmente el resumen del estado del solver y la información de diagnóstico disponible. Esto permite documentar el experimento y comparar posteriormente distintas configuraciones del problema, como variaciones en el número de tickets, la capacidad disponible o las restricciones activadas.

In [14]:
# ============================================================
# 14. EXPORTACIÓN DE RESULTADOS
# ============================================================

print("Exportando resultados a Google Drive...")


# ------------------------------------------------------------
# 14.1. Comprobación de la carpeta de salida
# ------------------------------------------------------------

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Carpeta de salida:")
print(OUTPUT_DIR)


# ------------------------------------------------------------
# 14.2. Identificador del experimento
# ------------------------------------------------------------

# El identificador permite diferenciar los resultados según
# el número de tickets utilizado en cada ejecución.

experiment_id = (
    f"cop_{len(ticket_ids)}_tickets"
)


# ------------------------------------------------------------
# 14.3. Exportación del resumen del solver
# ------------------------------------------------------------

solver_summary.to_csv(
    OUTPUT_DIR
    / f"{experiment_id}_solver_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


objective_summary.to_csv(
    OUTPUT_DIR
    / f"{experiment_id}_objective_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 14.4. Exportación del resumen de validación
# ------------------------------------------------------------

validation_summary.to_csv(
    OUTPUT_DIR
    / f"{experiment_id}_validation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


if (
    "team_capacity_df" in globals()
    and isinstance(team_capacity_df, pd.DataFrame)
    and not team_capacity_df.empty
):

    team_capacity_df.to_csv(
        OUTPUT_DIR
        / f"{experiment_id}_team_capacity.csv",
        index=False,
        encoding="utf-8-sig"
    )


# ------------------------------------------------------------
# 14.5. Exportación de la matriz de tiempos t_ij
# ------------------------------------------------------------

time_matrix_df.to_csv(
    OUTPUT_DIR
    / f"{experiment_id}_processing_time_matrix_tij.csv",
    index=True,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 14.6. Exportación de la matriz de elegibilidad
# ------------------------------------------------------------

eligibility_matrix_df.astype(int).to_csv(
    OUTPUT_DIR
    / f"{experiment_id}_eligibility_matrix.csv",
    index=True,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 14.7. Exportación del detalle de elegibilidad
# ------------------------------------------------------------

eligibility_df.to_csv(
    OUTPUT_DIR
    / f"{experiment_id}_eligibility_details.csv",
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 14.8. Exportación de los mecanismos de respaldo
# ------------------------------------------------------------

if (
    "fallback_df" in globals()
    and isinstance(fallback_df, pd.DataFrame)
):

    fallback_df.to_csv(
        OUTPUT_DIR
        / f"{experiment_id}_fallback_records.csv",
        index=False,
        encoding="utf-8-sig"
    )


# ------------------------------------------------------------
# 14.9. Exportación de la solución
# ------------------------------------------------------------

if solution_exists:

    assignments_df.to_csv(
        OUTPUT_DIR
        / f"{experiment_id}_assignments.csv",
        index=False,
        encoding="utf-8-sig"
    )

    employee_summary_df.to_csv(
        OUTPUT_DIR
        / f"{experiment_id}_employee_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    solution_summary_df.to_csv(
        OUTPUT_DIR
        / f"{experiment_id}_solution_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

else:

    # Aunque no exista una solución, se guarda el resumen
    # generado en la sección 13.

    solution_summary_df.to_csv(
        OUTPUT_DIR
        / f"{experiment_id}_infeasible_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )


# ------------------------------------------------------------
# 14.10. Registro general del experimento
# ------------------------------------------------------------

experiment_record = pd.DataFrame([{
    "experiment_id": experiment_id,
    "n_tickets": len(ticket_ids),
    "n_employees": len(employees),
    "solver_status": solver.StatusName(
        solver_status
    ),
    "solution_exists": solution_exists,
    "proven_optimal": proven_optimal,
    "beta_balance": BETA_BALANCE,
    "default_capacity_hours": (
        DEFAULT_CAPACITY_HOURS
    ),
    "priority_restriction": (
        ENFORCE_PRIORITY_HISTORY
    ),
    "team_restriction": ENFORCE_TEAM,
    "sla_restriction": ENFORCE_SLA,
    "solver_time_limit_seconds": (
        MAX_SOLVER_TIME_SECONDS
    ),
    "solver_runtime_seconds": (
        solver_runtime
    ),
    "assignment_variables": len(
        assignment_variables
    ),
    "fallback_tickets": len(
        fallback_records
    ),
    "minimum_required_capacity": (
        minimum_required_capacity
    ),
    "total_available_capacity": (
        total_available_capacity
    )
}])


experiment_record.to_csv(
    OUTPUT_DIR
    / f"{experiment_id}_experiment_record.csv",
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 14.11. Lista de archivos generados
# ------------------------------------------------------------

generated_files = sorted(
    OUTPUT_DIR.glob(
        f"{experiment_id}_*.csv"
    )
)


print("\n------------------------------------------")
print("ARCHIVOS EXPORTADOS")
print("------------------------------------------")

for file_path in generated_files:

    print("✓", file_path.name)


print(
    "\nResultados guardados correctamente en:"
)

print(OUTPUT_DIR)


# ------------------------------------------------------------
# 14.12. Resumen final
# ------------------------------------------------------------

if solution_exists:

    print(
        "\n✓ Se exportaron la solución, "
        "las asignaciones y los indicadores "
        "de carga de trabajo."
    )

else:

    print(
        "\n⚠ La instancia no dispone de una "
        "solución factible."
    )

    print(
        "Se exportaron los diagnósticos, "
        "el estado del solver y las matrices "
        "utilizadas para construir el problema."
    )

Exportando resultados a Google Drive...
Carpeta de salida:
/content/drive/MyDrive/TFM/resultados_cop

------------------------------------------
ARCHIVOS EXPORTADOS
------------------------------------------
✓ cop_200_tickets_eligibility_details.csv
✓ cop_200_tickets_eligibility_matrix.csv
✓ cop_200_tickets_experiment_record.csv
✓ cop_200_tickets_fallback_records.csv
✓ cop_200_tickets_infeasible_summary.csv
✓ cop_200_tickets_objective_summary.csv
✓ cop_200_tickets_processing_time_matrix_tij.csv
✓ cop_200_tickets_solver_summary.csv
✓ cop_200_tickets_team_capacity.csv
✓ cop_200_tickets_validation_summary.csv

Resultados guardados correctamente en:
/content/drive/MyDrive/TFM/resultados_cop

⚠ La instancia no dispone de una solución factible.
Se exportaron los diagnósticos, el estado del solver y las matrices utilizadas para construir el problema.


## 15. Análisis de rendimiento y escalabilidad

Finalmente, se presenta un resumen del rendimiento computacional del modelo y de las principales características de la instancia resuelta. El objetivo de esta sección es facilitar la comparación entre diferentes experimentos realizados con distintos tamaños del problema o diferentes configuraciones del modelo.

Los indicadores obtenidos permiten analizar cómo evoluciona el tiempo de resolución, el número de variables, el número de restricciones y el estado alcanzado por el solver a medida que aumenta el tamaño de la instancia. Esta información será utilizada posteriormente para evaluar la escalabilidad del enfoque propuesto y compararlo con otras técnicas de optimización consideradas en este trabajo, como los algoritmos genéticos.

In [15]:
# ============================================================
# 15. ANÁLISIS DE RENDIMIENTO Y ESCALABILIDAD
# ============================================================

print("Generando informe final...")


# ------------------------------------------------------------
# Estadísticas generales
# ------------------------------------------------------------

experiment_statistics = pd.DataFrame({

    "Indicador":[

        "Número de tickets",
        "Número de empleados",
        "Variables binarias",
        "Combinaciones posibles",
        "Combinaciones elegibles",
        "Reducción del espacio de búsqueda (%)",
        "Tiempo máximo del solver (s)",
        "Tiempo real del solver (s)",
        "Estado del solver",
        "Existe solución",
        "Optimalidad demostrada"

    ],

    "Valor":[

        len(ticket_ids),

        len(employees),

        len(assignment_variables),

        len(ticket_ids)*len(employees),

        len(assignment_variables),

        round(
            (
                1
                -
                len(assignment_variables)
                /
                (len(ticket_ids)*len(employees))
            )*100,
            2
        ),

        MAX_SOLVER_TIME_SECONDS,

        round(solver_runtime,4),

        solver.StatusName(solver_status),

        solution_exists,

        proven_optimal

    ]

})


display(experiment_statistics)

Generando informe final...


,Indicador,Valor
0,Número de tickets,200
1,Número de empleados,40
2,Variables binarias,1995
3,Combinaciones posibles,8000
4,Combinaciones elegibles,1995
5,Reducción del espacio de búsqueda (%),75.06
6,Tiempo máximo del solver (s),300
7,Tiempo real del solver (s),0.0058
8,Estado del solver,INFEASIBLE
9,Existe solución,False


In [16]:
# ------------------------------------------------------------
# Resumen final del experimento
# ------------------------------------------------------------

print("\n========================================")
print("EXPERIMENTO FINALIZADO")
print("========================================")

print(f"Tickets analizados: {len(ticket_ids)}")

print(f"Empleados: {len(employees)}")

print(f"Variables binarias: {len(assignment_variables)}")

print(f"Estado del solver: {solver.StatusName(solver_status)}")

print(f"Tiempo de resolución: {solver_runtime:.4f} segundos")


if solution_exists:

    print("\nSe obtuvo una solución.")

    if proven_optimal:

        print("La solución es óptima.")

    else:

        print("La solución es factible, aunque no se pudo demostrar su optimalidad.")

else:

    print("\nNo se encontró una solución factible.")

print("\nResultados exportados correctamente.")


EXPERIMENTO FINALIZADO
Tickets analizados: 200
Empleados: 40
Variables binarias: 1995
Estado del solver: INFEASIBLE
Tiempo de resolución: 0.0058 segundos

No se encontró una solución factible.

Resultados exportados correctamente.


## 12.7. Diagnóstico de inviabilidad y resumen de parámetros

Cuando el solver determina que una instancia es inviable, resulta necesario analizar qué condiciones del problema pueden estar provocando la ausencia de una solución factible. Para ello, se realiza un diagnóstico posterior que evalúa la capacidad total disponible, la carga mínima requerida, la capacidad por equipo, la existencia de tickets con un número muy reducido de candidatos y el efecto de las restricciones de equipo, experiencia histórica por prioridad y SLA.

Este diagnóstico no sustituye la demostración formal de inviabilidad realizada por CP-SAT, sino que permite identificar las causas operativas más probables que explican el resultado obtenido. Asimismo, se genera una tabla resumen con los principales parámetros utilizados en la ejecución, facilitando la documentación y comparación de distintos escenarios experimentales.

In [20]:
# ============================================================
# 12.7. DIAGNÓSTICO DE INVIABILIDAD Y TABLA DE PARÁMETROS
# ============================================================

print("\n==========================================")
print("DIAGNÓSTICO DEL EXPERIMENTO")
print("==========================================\n")


# ------------------------------------------------------------
# 1. TABLA DE PARÁMETROS DEL EXPERIMENTO
# ------------------------------------------------------------

sla_p1 = SLA_BY_PRIORITY.get("1", SLA_BY_PRIORITY.get("P1", np.nan))
sla_p2 = SLA_BY_PRIORITY.get("2", SLA_BY_PRIORITY.get("P2", np.nan))
sla_p3 = SLA_BY_PRIORITY.get("3", SLA_BY_PRIORITY.get("P3", np.nan))
sla_p4 = SLA_BY_PRIORITY.get("4", SLA_BY_PRIORITY.get("P4", np.nan))


parameter_summary_df = pd.DataFrame({
    "Parámetro": [
        "Número de tickets",
        "Número de empleados",
        "Combinaciones teóricas",
        "Variables binarias creadas",
        "Capacidad base por empleado",
        "Capacidad total disponible",
        "SLA P1",
        "SLA P2",
        "SLA P3",
        "SLA P4",
        "Restricción de equipo",
        "Restricción de experiencia por prioridad",
        "Mínimo histórico por prioridad",
        "Restricción de SLA",
        "Beta de balance",
        "Escala temporal",
        "Tiempo máximo del solver",
        "Procesadores del solver",
        "Semilla aleatoria",
        "Estado obtenido",
        "Tiempo real de resolución"
    ],
    "Valor": [
        len(ticket_ids),
        len(employees),
        len(ticket_ids) * len(employees),
        len(assignment_variables),
        f"{DEFAULT_CAPACITY_HOURS:.2f} h",
        f"{sum(employee_capacity.values()):.2f} h",
        f"{sla_p1} h" if pd.notna(sla_p1) else "No definido",
        f"{sla_p2} h" if pd.notna(sla_p2) else "No definido",
        f"{sla_p3} h" if pd.notna(sla_p3) else "No definido",
        f"{sla_p4} h" if pd.notna(sla_p4) else "No definido",
        ENFORCE_TEAM,
        ENFORCE_PRIORITY_HISTORY,
        MIN_PRIORITY_HISTORY,
        ENFORCE_SLA,
        BETA_BALANCE,
        TIME_SCALE,
        f"{MAX_SOLVER_TIME_SECONDS:.2f} s",
        NUM_SEARCH_WORKERS,
        RANDOM_SEED,
        solver_status_name,
        f"{solver_runtime:.4f} s"
    ]
})


print("TABLA DE PARÁMETROS DEL EXPERIMENTO\n")

display(parameter_summary_df)


DIAGNÓSTICO DEL EXPERIMENTO

TABLA DE PARÁMETROS DEL EXPERIMENTO



,Parámetro,Valor
0,Número de tickets,200
1,Número de empleados,40
2,Combinaciones teóricas,8000
3,Variables binarias creadas,1995
4,Capacidad base por empleado,40.00 h
5,Capacidad total disponible,1600.00 h
6,SLA P1,8.0 h
7,SLA P2,16.0 h
8,SLA P3,24.0 h
9,SLA P4,40.0 h


In [18]:
# ------------------------------------------------------------
# 2. DIAGNÓSTICO DE INVIABILIDAD
# ------------------------------------------------------------

infeasibility_findings = []


# ------------------------------------------------------------
# A. CAPACIDAD TOTAL
# ------------------------------------------------------------

total_capacity = sum(
    employee_capacity.values()
)

minimum_required = sum(
    minimum_time_by_ticket.values()
)


if minimum_required > total_capacity:

    infeasibility_findings.append({
        "Categoría": "Capacidad total",
        "Resultado": "FALLO",
        "Detalle": (
            f"La carga mínima estimada necesaria es "
            f"{minimum_required:.2f} h, mientras que la "
            f"capacidad total disponible es "
            f"{total_capacity:.2f} h. "
            f"Existe un déficit mínimo de "
            f"{minimum_required - total_capacity:.2f} h."
        )
    })

else:

    infeasibility_findings.append({
        "Categoría": "Capacidad total",
        "Resultado": "OK",
        "Detalle": (
            f"La capacidad total disponible "
            f"({total_capacity:.2f} h) es superior o igual "
            f"a la carga mínima estimada "
            f"({minimum_required:.2f} h)."
        )
    })


# ------------------------------------------------------------
# B. CAPACIDAD POR EQUIPO
# ------------------------------------------------------------

teams_over_capacity = []

if "team_capacity_df" in globals():

    for _, row in team_capacity_df.iterrows():

        required = row[
            "minimum_required_hours"
        ]

        available = row[
            "available_capacity_hours"
        ]

        if required > available:

            teams_over_capacity.append(
                (
                    row["team"],
                    required,
                    available
                )
            )


if teams_over_capacity:

    detail = "; ".join(
        [
            (
                f"{team}: requiere mínimo "
                f"{required:.2f} h y dispone de "
                f"{available:.2f} h"
            )
            for team, required, available
            in teams_over_capacity
        ]
    )

    infeasibility_findings.append({
        "Categoría": "Capacidad por equipo",
        "Resultado": "FALLO",
        "Detalle": detail
    })

else:

    infeasibility_findings.append({
        "Categoría": "Capacidad por equipo",
        "Resultado": "OK",
        "Detalle": (
            "No se detectaron equipos cuya carga mínima "
            "estimada supere su capacidad disponible."
        )
    })


# ------------------------------------------------------------
# C. TICKETS SIN CANDIDATOS ANTES DEL FALLBACK
# ------------------------------------------------------------

originally_without_candidates = []

for ticket in ticket_ids:

    ticket_rows = eligibility_df[
        eligibility_df["ticket"] == ticket
    ]

    originally_eligible = ticket_rows[
        (
            ticket_rows["team_ok"]
            &
            ticket_rows["priority_ok"]
            &
            ticket_rows["sla_ok"]
        )
    ]

    if len(originally_eligible) == 0:

        originally_without_candidates.append(
            ticket
        )


if originally_without_candidates:

    infeasibility_findings.append({
        "Categoría": "Elegibilidad",
        "Resultado": "ADVERTENCIA",
        "Detalle": (
            f"{len(originally_without_candidates)} tickets "
            f"no tenían ningún empleado que cumpliera "
            f"simultáneamente equipo, prioridad y SLA "
            f"antes de aplicar el mecanismo de respaldo."
        )
    })

else:

    infeasibility_findings.append({
        "Categoría": "Elegibilidad",
        "Resultado": "OK",
        "Detalle": (
            "Todos los tickets tenían al menos un empleado "
            "que cumplía las restricciones activas."
        )
    })


# ------------------------------------------------------------
# D. EFECTO DE LA RESTRICCIÓN DE EQUIPO
# ------------------------------------------------------------

team_failures = int(
    (~eligibility_df["team_ok"]).sum()
)

total_combinations = len(
    eligibility_df
)

team_failure_percentage = (
    100 * team_failures / total_combinations
    if total_combinations > 0
    else 0
)


infeasibility_findings.append({
    "Categoría": "Restricción de equipo",
    "Resultado": (
        "ACTIVA"
        if ENFORCE_TEAM
        else "INACTIVA"
    ),
    "Detalle": (
        f"{team_failures} combinaciones "
        f"({team_failure_percentage:.2f} %) "
        f"no cumplen la compatibilidad de equipo."
    )
})


# ------------------------------------------------------------
# E. EFECTO DE LA EXPERIENCIA POR PRIORIDAD
# ------------------------------------------------------------

priority_failures = int(
    (~eligibility_df["priority_ok"]).sum()
)

priority_failure_percentage = (
    100 * priority_failures / total_combinations
    if total_combinations > 0
    else 0
)


infeasibility_findings.append({
    "Categoría": "Experiencia por prioridad",
    "Resultado": (
        "ACTIVA"
        if ENFORCE_PRIORITY_HISTORY
        else "INACTIVA"
    ),
    "Detalle": (
        f"{priority_failures} combinaciones "
        f"({priority_failure_percentage:.2f} %) "
        f"no cumplen el histórico mínimo exigido."
    )
})


# ------------------------------------------------------------
# F. EFECTO DEL SLA
# ------------------------------------------------------------

sla_failures = int(
    (~eligibility_df["sla_ok"]).sum()
)

sla_failure_percentage = (
    100 * sla_failures / total_combinations
    if total_combinations > 0
    else 0
)


infeasibility_findings.append({
    "Categoría": "Restricción SLA",
    "Resultado": (
        "ACTIVA"
        if ENFORCE_SLA
        else "INACTIVA"
    ),
    "Detalle": (
        f"{sla_failures} combinaciones "
        f"({sla_failure_percentage:.2f} %) "
        f"presentan un tiempo estimado superior al SLA."
    )
})


# ------------------------------------------------------------
# G. TICKETS CON MUY POCOS CANDIDATOS
# ------------------------------------------------------------

candidate_counts = {
    ticket: len(
        eligible_employees_by_ticket[
            ticket
        ]
    )
    for ticket in ticket_ids
}


tickets_one_candidate = [
    ticket
    for ticket, count
    in candidate_counts.items()
    if count == 1
]


tickets_two_or_less = [
    ticket
    for ticket, count
    in candidate_counts.items()
    if count <= 2
]


infeasibility_findings.append({
    "Categoría": "Concentración de elegibilidad",
    "Resultado": "DIAGNÓSTICO",
    "Detalle": (
        f"{len(tickets_one_candidate)} tickets tienen "
        f"un único empleado elegible y "
        f"{len(tickets_two_or_less)} tickets tienen "
        f"como máximo dos candidatos."
    )
})


# ------------------------------------------------------------
# H. RESULTADO DEL SOLVER
# ------------------------------------------------------------

if solver_status == cp_model.INFEASIBLE:

    solver_detail = (
        "CP-SAT demostró que no existe ninguna asignación "
        "capaz de satisfacer simultáneamente todas las "
        "restricciones activas."
    )

else:

    solver_detail = (
        f"El solver obtuvo estado {solver_status_name}."
    )


infeasibility_findings.append({
    "Categoría": "Resultado formal CP-SAT",
    "Resultado": solver_status_name,
    "Detalle": solver_detail
})


# ------------------------------------------------------------
# TABLA FINAL
# ------------------------------------------------------------

infeasibility_analysis_df = pd.DataFrame(
    infeasibility_findings
)


print("\nANÁLISIS DE FACTIBILIDAD / INVIABILIDAD\n")

display(
    infeasibility_analysis_df
)


ANÁLISIS DE FACTIBILIDAD / INVIABILIDAD



,Categoría,Resultado,Detalle
0,Capacidad total,FALLO,La carga mínima estimada necesaria es 1633.89 ...
1,Capacidad por equipo,FALLO,CS: requiere mínimo 1068.93 h y dispone de 720...
2,Elegibilidad,ADVERTENCIA,15 tickets no tenían ningún empleado que cumpl...
3,Restricción de equipo,ACTIVA,5248 combinaciones (65.60 %) no cumplen la com...
4,Experiencia por prioridad,ACTIVA,761 combinaciones (9.51 %) no cumplen el histó...
5,Restricción SLA,ACTIVA,1570 combinaciones (19.62 %) presentan un tiem...
6,Concentración de elegibilidad,DIAGNÓSTICO,29 tickets tienen un único empleado elegible y...
7,Resultado formal CP-SAT,INFEASIBLE,CP-SAT demostró que no existe ninguna asignaci...


In [19]:
# ------------------------------------------------------------
# 3. INTERPRETACIÓN AUTOMÁTICA
# ------------------------------------------------------------

print("\n==========================================")
print("INTERPRETACIÓN DEL DIAGNÓSTICO")
print("==========================================\n")


if solver_status != cp_model.INFEASIBLE:

    print(
        "La instancia no ha sido declarada inviable "
        "por CP-SAT."
    )

else:

    probable_causes = []


    # Capacidad total
    if minimum_required > total_capacity:

        probable_causes.append(
            "la capacidad total disponible es inferior "
            "a la carga mínima necesaria"
        )


    # Capacidad por equipo
    if teams_over_capacity:

        probable_causes.append(
            "uno o varios equipos presentan una carga "
            "mínima superior a su capacidad disponible"
        )


    # Tickets originalmente sin candidatos
    if originally_without_candidates:

        probable_causes.append(
            "existen tickets sin candidatos que cumplan "
            "simultáneamente equipo, prioridad y SLA"
        )


    # Concentración excesiva
    if len(tickets_one_candidate) > 0:

        probable_causes.append(
            "existen tickets cuya asignación está "
            "restringida a un único empleado"
        )


    if probable_causes:

        print(
            "La inviabilidad parece estar asociada "
            "principalmente a:"
        )

        for cause in probable_causes:
            print("•", cause)

    else:

        print(
            "No se detectó una causa simple de inviabilidad "
            "mediante los diagnósticos agregados."
        )

        print(
            "La inviabilidad probablemente deriva de la "
            "interacción simultánea entre las restricciones "
            "de capacidad y elegibilidad."
        )


INTERPRETACIÓN DEL DIAGNÓSTICO

La inviabilidad parece estar asociada principalmente a:
• la capacidad total disponible es inferior a la carga mínima necesaria
• uno o varios equipos presentan una carga mínima superior a su capacidad disponible
• existen tickets sin candidatos que cumplan simultáneamente equipo, prioridad y SLA
• existen tickets cuya asignación está restringida a un único empleado
